# Adaptive-K GRPO on HotpotQA (APEX-GRPO Architecture v2) — Google Colab

Run setup and definition cells in order. Set the operation switches near the end only when ready. Defaults match the Static-K Colab notebook: seed 42; 200 training episodes; 50 validation / 200 held-out test examples; Ollama `llama3.1:8b` execution LLM; Hugging Face `hotpotqa/hotpot_qa` (distractor split).

This **v2 notebook** implements the complete **4-Pillar APEX-GRPO Architecture** from `adaptive_rollout_plan.md`:
1. **Pillar 1: Analytical Pre-Allocation (Layer B):** Closed-form coverage $\mathbb{E}[U(K)]$ and penalized utility $\mathcal{J}(K)$ setting $K_{\text{ceiling}} \in \{5, 8, 10, 15\}$ with GVM-RAFT difficulty regularization.
2. **Pillar 2: Intra-Episode Deduplication & Deterministic Caching (Layer A):** Downstream LLM inference at `temperature = 0.0` is executed only once per unique pipeline; duplicate candidates reuse cached results.
3. **Pillar 3: Contrast-Based Early Exit & Failure-Only Rescue (Layer C):** Staged waterfall ($5 \to 8 \to 10 \to 15$) enforcing $K \le K_{\text{ceiling}}$ and the All-Correct Invariant.
4. **Pillar 4: Zero-Gradient Optimizer Masking (Layer D):** Skips Adam parameter updates on uninformative zero-variance groups ($A_i = 0$), preserving momentum buffers.
5. **Exact 1/K Loss Normalization:** Normalizes Figure 2 GRPO surrogate loss by group size $\frac{1}{K}$.


In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")

CACHE = Path("/content/drive/MyDrive/GRPO_cache")
CACHE.mkdir(parents=True, exist_ok=True)

# Reuse Python package downloads.
os.environ["PIP_CACHE_DIR"] = str(CACHE / "pip")

# Store Ollama models persistently.
# Must be set BEFORE starting "ollama serve".
os.environ["OLLAMA_MODELS"] = str(CACHE / "ollama_models")

In [ ]:
import os
import time
import subprocess
import urllib.request

print("=== Starting Colab Environment Setup ===")

# 1. Install system dependencies
print("Installing zstd...")
subprocess.run("sudo apt-get update -qq && sudo apt-get install -y zstd", shell=True, check=True)

# 2. Install Ollama
print("Installing Ollama...")
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

# 3. Start Ollama daemon in the background and wait until it is ready
print("Starting Ollama serve...")
ollama_process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Poll the host endpoint until it responds
ready = False
for i in range(30):
    try:
        with urllib.request.urlopen("http://localhost:11434/") as response:
            if response.status == 200:
                ready = True
                break
    except Exception:
        time.sleep(2)

if ready:
    print("Ollama daemon is up and running!")
else:
    raise RuntimeError("Ollama daemon failed to start.")

# 4. Pull the model specified in your CONFIG
TARGET_MODEL = "llama3.1:8b"
print(f"Pulling Ollama model: {TARGET_MODEL} (this may take a moment)...")
check = subprocess.run(
    ["ollama", "show", TARGET_MODEL],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

if check.returncode != 0:
    subprocess.run(["ollama", "pull", TARGET_MODEL], check=True)
else:
    print("Using cached model:", TARGET_MODEL)
print("Setup complete and verified!")

In [ ]:
!ollama list

# 2. Environment and Dependency Setup
Use the existing GPU virtual environment as your Jupyter kernel. Installation is opt-in and does not replace your PyTorch build. Install CUDA-compatible PyTorch separately if needed. Version snapshots accompany every run; frozen versions alone do not guarantee Ollama determinism.

In [ ]:
%pip install -q dspy==3.3.0 transformers==5.14.1 datasets==5.0.1 "numpy>=2.0,<3" "matplotlib>=3.10,<4" "nbformat>=5.10" ipykernel

The setup above follows the Static-K Colab notebook and downloads packages and `llama3.1:8b`. Keep the runtime active while Ollama runs. Drive stores cached models and backup copies; active training and validation run on local disk for speed.

# Adaptive-K GRPO: HotpotQA (APEX-GRPO v2)

This notebook implements the **APEX-GRPO** adaptive rollout optimization framework for AutoDSPy on the HotpotQA multi-hop question answering benchmark:
- **Discrete 30-Pipeline Action Space:** AutoDSPy operates over a compact action space of 2 modules $\times$ 15 signatures.
- **Pillar 1 (Analytical Pre-Allocation — Layer B):** Before dispatching rollouts, GPT-2 computes exact categorical probabilities $\pi_j$ over all 30 pipelines. Solves $\max_K [\mathbb{E}[U(K)] - \lambda K]$ ($\lambda=0.25$) to determine $K_{\text{ceiling}} \in \{5, 8, 10, 15\}$. Consistently unsolvable prompts ($\hat{p}_x \approx 0$) are capped at $K=5$.
- **Pillar 2 (Intra-Episode Deduplication & Caching — Layer A):** If duplicate pipelines are drawn within a group, Llama-3.1 executes only once at `temperature = 0.0`; cached outputs/rewards are reused while preserving exact group statistics.
- **Pillar 3 (Contrast-Based Early Exit & Failure-Only Rescue — Layer C):** Candidates are evaluated in deterministic stages ($5 \to 8 \to 10 \to 15$). Execution exits immediately upon finding reward contrast ($\max(r) - \min(r) > 0$) or when all candidates succeed ($5/5$ correct). Expansion only occurs on all-failure groups when $K_{\text{ceiling}}$ permits.
- **Pillar 4 (Zero-Gradient Optimizer Masking — Layer D):** When group advantages are zero ($A_i = 0$), `optimizer.step()` is skipped to prevent Adam momentum drift.
- **Loss Normalization:** All GRPO surrogate losses are normalized by $1/K$.

**Saved outputs follow immutable protocols. Restart kernel and rerun definition cells in order.**


# 2. Environment and Dependency Setup
Use the existing GPU virtual environment as your Jupyter kernel. Installation is opt-in and does not replace your PyTorch build. Install CUDA-compatible PyTorch separately if needed. Version snapshots accompany every run; frozen versions alone do not guarantee Ollama determinism.

In [ ]:
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "dspy==3.3.0", "transformers==5.14.1",
                           "datasets==5.0.1", "numpy>=2.0,<3", "matplotlib>=3.10,<4", "nbformat>=5.10", "ipykernel"])

import copy, gc, hashlib, importlib.metadata, json, math, os, random, re, sys, tempfile, time, uuid
from contextlib import contextmanager
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation
from pathlib import Path
from urllib.request import Request, urlopen
import numpy as np
import torch
import dspy
import matplotlib.pyplot as plt
from datasets import load_dataset
from huggingface_hub import HfApi
from transformers import GPT2LMHeadModel, GPT2Tokenizer

VERSIONS = {name: importlib.metadata.version(name) for name in
            ["torch", "transformers", "dspy", "datasets", "numpy", "matplotlib"]}
print("Python:", sys.version, "\nPackages:", VERSIONS, "\nCUDA:", torch.cuda.is_available())

# 3. Experiment Configuration
Edit these controls, then call `build_config()` again. Smoke/pilot/baseline default to 5/10/200 episodes using the same 200-example pool and persisted 200-episode schedule. Smaller explicit sizes are engineering checks. GPT-2 (paper label 127M), Adam 1e-4, and baseline K=5 follow reported settings; K=8, seed, subsets and resource/decoding settings are project choices.

`OLLAMA_NUM_GPU=0` offloads Llama to CPU, useful when GPT-2 and its optimizer share a 6GB GPU. Change it explicitly for partial/automatic offload; keep it identical across compared K runs. No model substitution occurs.
**Reward mode:** Choose `original_semantic` (default) or `answer_exact`, then rerun the configuration cell to rebuild `CONFIG`.

In [ ]:
REWARD_MODE = "original_semantic"       # original_semantic / answer_exact
JUDGE_CACHE_ENABLED = True
JUDGE_LOGGING_ENABLED = True
LEGACY_REWARD_PROTOCOL = False         # True reproduces the uncached historical reward mechanism

SEED = 42
PRESET = "baseline"                       # smoke / pilot / baseline
MODE = "initialize"                    # initialize / train / resume / validate / test / compare
NUM_EPISODES = None                     # None chooses preset; override freely
TRAIN_POOL_SIZE = 200
VALIDATION_SIZE = 50
TEST_SIZE = 200
OLLAMA_MODEL = "llama3.1:8b"
OLLAMA_HOST = "http://localhost:11434"
CHECKPOINT_INTERVAL = 10
VALIDATION_INTERVAL = None              # baseline 50; smoke 5; pilot 10
SCHEDULE_EPISODES = 200                 # common maximum; presets use prefixes
POLICY_MODEL = "gpt2"
POLICY_REVISION = "607a30d783dfa663caf39e06633721c8d4cfcd7e"
DATASET_REVISION = None                 # resolve main to immutable HF commit before loading
LEARNING_RATE = 1e-5
POLICY_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OLLAMA_NUM_GPU = None                      # 0=CPU; None=Ollama default; integer=GPU layers
OLLAMA_NUM_CTX = 4096
EXECUTION_MAX_TOKENS = 512
EXECUTION_TEMPERATURE = 0.0
REQUEST_TIMEOUT = 180
RESULTS_ROOT = Path("/content/adaptive_grpo_results")
SELECTED_CHECKPOINT = None              # explicit path for resume / validate / test
RUN_OPERATION = False                  # section 21 safety switch
RUN_ALL_K = False                      # explicit sequential fresh starts; never auto-enabled
ALLOWED_K = (5, 8, 10, 15)
PROTOCOL_VERSION = "hotpotqa-apex-adaptive-rollout-v2"

# APEX-GRPO 4-Pillar Adaptive Rollout Configuration
ADAPTIVE_LEVELS = (5, 8, 10, 15)
CANDIDATE_BUDGET = None  # hard cap on reserved training attempts, including rollback; None disables
RESCUE_MODE = "failure_only"  # Failure-only rescue per Section 6.2 invariant
REWARD_CONTRAST = 0.0  # minimum difference; 0 means any positive difference
LAMBDA_PENALTY = 0.25  # Marginal efficiency penalty (lambda = 0.25)
ALLOCATION_VERSION = "apex-adaptive-rollout-v2"

def build_config(**overrides):
    presets = {"smoke": 5, "pilot": 10, "baseline": 200}
    if PRESET not in presets:
        raise ValueError("PRESET must be smoke, pilot or baseline")
    cfg = dict(dataset="hotpotqa", dataset_name="hotpotqa/hotpot_qa", dataset_config="distractor",
               dataset_revision=DATASET_REVISION, policy_model=POLICY_MODEL, policy_revision=POLICY_REVISION,
               k=max(ADAPTIVE_LEVELS), seed=SEED, preset=PRESET, num_episodes=NUM_EPISODES if NUM_EPISODES is not None else presets[PRESET],
               train_pool_size=TRAIN_POOL_SIZE, validation_size=VALIDATION_SIZE, test_size=TEST_SIZE,
               schedule_episodes=SCHEDULE_EPISODES, learning_rate=LEARNING_RATE,
               checkpoint_interval=CHECKPOINT_INTERVAL,
               validation_interval=VALIDATION_INTERVAL if VALIDATION_INTERVAL is not None else {"smoke":5,"pilot":10,"baseline":50}[PRESET],
               policy_device=POLICY_DEVICE, ollama_model=OLLAMA_MODEL, ollama_host=OLLAMA_HOST.rstrip("/"),
               ollama_num_gpu=OLLAMA_NUM_GPU, ollama_num_ctx=OLLAMA_NUM_CTX,
               execution_max_tokens=EXECUTION_MAX_TOKENS, execution_temperature=EXECUTION_TEMPERATURE,
               request_timeout=REQUEST_TIMEOUT, results_root=str(RESULTS_ROOT), protocol=PROTOCOL_VERSION,
               policy_dropout=False, policy_sampling="unique-signature-token-label-v1",
               loss="negative-sum-logprob-times-centered-reward", optimizer="Adam", cache=False,
               semantic_fallback=REWARD_MODE == "original_semantic", reward_mode=REWARD_MODE, max_policy_tokens=1024, max_pipeline_steps=3,
               evaluation_sampling="multinomial-with-fixed-split-seed", evaluation_seed_offset=100000,
               adaptive_levels=list(ADAPTIVE_LEVELS), rescue_mode=RESCUE_MODE,
               reward_contrast=REWARD_CONTRAST, candidate_budget=CANDIDATE_BUDGET,
               lambda_penalty=LAMBDA_PENALTY, allocation_version=ALLOCATION_VERSION,
               protocol_version=PROTOCOL_VERSION, normalize_loss_by_k=True)
    cfg.update(overrides)
    if cfg["reward_mode"] not in ("original_semantic", "answer_exact"):
        raise ValueError("reward_mode must be original_semantic or answer_exact")
    cfg["semantic_fallback"] = cfg["reward_mode"] == "original_semantic"
    if cfg["k"] not in ALLOWED_K:
        raise ValueError(f"K must be one of {ALLOWED_K}")
    for key in ["num_episodes","train_pool_size","validation_size","test_size","schedule_episodes",
                "checkpoint_interval","validation_interval","execution_max_tokens","ollama_num_ctx"]:
        if not isinstance(cfg[key], int) or cfg[key] <= 0:
            raise ValueError(f"{key} must be a positive integer")
    cfg["schedule_episodes"] = max(cfg["schedule_episodes"], cfg["num_episodes"])
    if cfg["learning_rate"] <= 0 or cfg["policy_device"] not in ("cpu", "cuda"):
        raise ValueError("Use positive learning rate and cpu/cuda policy_device")
    levels = cfg["adaptive_levels"]
    if (len(levels) < 1 or any(type(x) is not int or x < 2 for x in levels)
            or levels != sorted(set(levels)) or levels[-1] > cfg["k"]):
        raise ValueError("Adaptive levels must be increasing integers in [2, K]")
    if cfg["rescue_mode"] not in ("both", "failure_only"):
        raise ValueError("rescue mode must be both or failure_only")
    if not math.isfinite(cfg["reward_contrast"]) or not 0 <= cfg["reward_contrast"] < 1:
        raise ValueError("reward contrast must be in [0,1)")
    if not math.isfinite(cfg["lambda_penalty"]) or cfg["lambda_penalty"] <= 0:
        raise ValueError("lambda_penalty must be a positive finite float")
    return cfg

def adaptive_decision(rewards, failures, cfg, k_ceiling=None):
    """
    Pillar 3: Evaluates staged stopping decisions at level boundaries {5, 8, 10, 15}.
    Enforces:
    - Priority Invariant: Stop if k_ceiling is reached.
    - All-Correct Invariant: Stop immediately if min(rewards) == 1.0 (failure-only rescue).
    - Contrast Exit: Stop immediately if max(rewards) - min(rewards) > cfg['reward_contrast'].
    - Staged Expansion: Only advance if all candidates failed (max(rewards) == 0.0)
      AND k_ceiling permits higher allocation.
    """
    current_k = len(rewards)
    levels = cfg["adaptive_levels"]
    if current_k not in levels:
        return None
    if failures:
        return "failure_guard"
    values = np.asarray(rewards, dtype=float)
    if not np.isfinite(values).all():
        raise ValueError("Nonfinite reward")
    
    # 1. Reward contrast observed -> exit immediately
    if float(values.max() - values.min()) > cfg["reward_contrast"]:
        return "contrast"
        
    # 2. All-correct invariant: all candidates succeeded -> exit immediately (skip update via Pillar 4)
    if float(values.min()) == 1.0:
        return "all_correct_exit"
        
    # 3. Non-failure homogeneous (e.g. all 0.5 under continuous/fractional rewards)
    if cfg.get("rescue_mode") == "failure_only" and not np.all(values == 0):
        return "non_failure_homogeneous"
        
    # 4. Ceiling and maximum checks
    effective_ceiling = k_ceiling if k_ceiling is not None else levels[-1]
    if current_k >= effective_ceiling:
        return "ceiling_reached"
    if current_k == levels[-1]:
        return "maximum_k"
        
    return None

STATIC_RUN_CONFIG_PATH = None
STATIC_REFERENCE = None
if STATIC_RUN_CONFIG_PATH:
    STATIC_REFERENCE = json.loads(Path(STATIC_RUN_CONFIG_PATH).read_text(encoding="utf-8"))
    reference_config = STATIC_REFERENCE["config"]
    required = ("dataset_revision", "policy_revision", "policy_model", "ollama_model", "seed",
                "train_pool_size", "validation_size", "test_size", "schedule_episodes")
    if any(reference_config.get(key) is None for key in required):
        raise ValueError("Static run config is missing a resolved dataset, model, or selection setting")
    DATASET_REVISION = reference_config["dataset_revision"]
    POLICY_REVISION = reference_config["policy_revision"]
    POLICY_MODEL = reference_config["policy_model"]
    OLLAMA_MODEL = reference_config["ollama_model"]
    SEED = reference_config["seed"]
    TRAIN_POOL_SIZE = reference_config["train_pool_size"]
    VALIDATION_SIZE = reference_config["validation_size"]
    TEST_SIZE = reference_config["test_size"]
    SCHEDULE_EPISODES = reference_config["schedule_episodes"]
    print("Pinned reference dataset/model settings from", STATIC_RUN_CONFIG_PATH)
else:
    print("No static run pinned: exact dataset revision/selection must be verified before comparison.")

CONFIG = build_config()
print(CONFIG)


### Drive backup and recovery

Checkpoints and their logs are copied to `MyDrive/Adaptive_K_GRPO_HotpotQA_v2` after every checkpoint and again after checkpoint summaries, tests, and plots. All checkpoints are retained. A completed-marker manifest protects against incomplete uploads; the reference workflow validates file sizes, not cryptographic hashes. Avoid simultaneous runtimes using the same backup directory.

After a runtime reset, rerun setup and configuration, set `RESTORE_FROM_DRIVE = True` in the next cell, and run it with an empty `/content/adaptive_grpo_results`. Set it back to `False` afterward. Run all definition cells, point `RUN_DIR` at the restored run, and explicitly run the resume cell. Keep package versions, Ollama version/model digest, dataset revision, and the original results path unchanged. Never use the fresh-training cell to resume.

In [ ]:
# Colab Drive backup controls. Active results stay on local disk.
from pathlib import Path
import shutil
import json
from datetime import datetime, timezone


BACKUP_TO_DRIVE = True
RESTORE_FROM_DRIVE = False  # Set True only in a fresh runtime before resuming.
DRIVE_BACKUP_DIR = Path("/content/drive/MyDrive/Adaptive_K_GRPO_HotpotQA_v2")

import shutil
try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    # Drive was mounted by the first setup cell.

def copy_backup_file(source, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".uploading")
    shutil.copyfile(source, temporary)
    temporary.replace(destination)

def backup_results(cfg=None):
    if not IN_COLAB or not BACKUP_TO_DRIVE:
        return
    local = RESULTS_ROOT.resolve()
    if cfg is not None and Path(cfg["results_root"]).resolve() != local:
        return  # Do not back up synthetic test fixtures.
    if not local.is_dir():
        return
    destination = DRIVE_BACKUP_DIR / "results"
    remote = destination.resolve()
    if local == remote or local in remote.parents or remote in local.parents:
        raise ValueError("Keep local RESULTS_ROOT separate from the Drive backup")
    DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
    marker = DRIVE_BACKUP_DIR / "backup_complete.json"
    marker.unlink(missing_ok=True)
    files = {}
    for item in sorted(local.rglob("*")):
        if not item.is_file() or item.name.endswith((".tmp", ".uploading")):
            continue
        relative = item.relative_to(local)
        target = destination / relative
        size = item.stat().st_size
        # Checkpoints are immutable; always refresh mutable logs and pointers.
        if item.suffix != ".pt" or not target.exists() or target.stat().st_size != size:
            copy_backup_file(item, target)
        files[relative.as_posix()] = size
    metadata = dict(local_root=str(local), files=files,
                    saved_at=datetime.now(timezone.utc).isoformat())
    temporary = marker.with_suffix(".uploading")
    temporary.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    temporary.replace(marker)
    print("Drive backup complete:", DRIVE_BACKUP_DIR)

def restore_results():
    if not IN_COLAB:
        raise RuntimeError("Run Drive recovery inside Colab")
    marker = DRIVE_BACKUP_DIR / "backup_complete.json"
    if not marker.is_file():
        raise RuntimeError("No complete backup found; an upload may have been interrupted")
    metadata = json.loads(marker.read_text(encoding="utf-8"))
    local = RESULTS_ROOT.resolve()
    if str(local) != metadata["local_root"]:
        raise ValueError("Use the original working directory and RESULTS_ROOT: " + metadata["local_root"])
    if local.exists() and any(local.iterdir()):
        raise FileExistsError("Restore only into an empty results folder; existing results are protected")
    remote = DRIVE_BACKUP_DIR / "results"
    for relative, size in metadata["files"].items():
        item, target = remote / relative, local / relative
        if not target.resolve().is_relative_to(local) or not item.resolve().is_relative_to(remote.resolve()):
            raise ValueError("Invalid backup path")
        if not item.is_file() or item.stat().st_size != size:
            raise RuntimeError("Incomplete backup file: " + relative)
    for relative in metadata["files"]:
        copy_backup_file(remote / relative, local / relative)
    print("Restored local results:", local)

if RESTORE_FROM_DRIVE:
    restore_results()


# 4. Hardware and Ollama Initialization
No server is launched or model pulled automatically. Start Ollama yourself and run `ollama pull llama3.1:8b` if absent. The endpoint is configurable, including a remote machine.

The instrumented original `dspy.LM('ollama_chat/...')` counts LM **forward attempts**, including adapter retries and semantic judges; these are logical calls. Exact HTTP request counts are unavailable because provider internals are not instrumented. Cache is disabled and provider retries are zero. Returned token usage is measured when present; failures may have unreported tokens. Ollama `/api/ps` reports server model residency, distinct from PyTorch allocator memory.

In [ ]:
def json_request(host, route, payload=None, timeout=15):
    data = None if payload is None else json.dumps(payload).encode()
    request = Request(host.rstrip("/")+route, data=data, headers={"Content-Type":"application/json"})
    with urlopen(request, timeout=timeout) as response:
        return json.load(response)

def ollama_snapshot(cfg):
    try:
        return json_request(cfg["ollama_host"], "/api/ps", timeout=5)
    except Exception as exc:
        return {"unavailable": f"{type(exc).__name__}: {exc}"}

class MeasuredLM(dspy.LM):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.events = []
    def forward(self, *args, **kwargs):
        event = {"seconds": None, "prompt_tokens": None, "generated_tokens": None, "error": None}
        started = time.perf_counter()
        try:
            response = super().forward(*args, **kwargs)
            usage = dict(getattr(response, "usage", {}) or {})
            event["prompt_tokens"] = usage.get("prompt_tokens")
            event["generated_tokens"] = usage.get("completion_tokens")
            return response
        except Exception as exc:
            event["error"] = f"{type(exc).__name__}: {exc}"
            raise
        finally:
            event["seconds"] = time.perf_counter()-started
            self.events.append(event)

def initialize_ollama(cfg):
    try:
        version = json_request(cfg["ollama_host"], "/api/version")
        tags = json_request(cfg["ollama_host"], "/api/tags")["models"]
        model = next((m for m in tags if m.get("name")==cfg["ollama_model"] or m.get("model")==cfg["ollama_model"]), None)
        if model is None:
            raise RuntimeError(f"Missing {cfg['ollama_model']}; run ollama pull {cfg['ollama_model']}")
        show = json_request(cfg["ollama_host"], "/api/show", {"model":cfg["ollama_model"]})
    except Exception as exc:
        raise RuntimeError(f"Ollama unavailable at {cfg['ollama_host']}: {exc}") from exc
    options = dict(api_base=cfg["ollama_host"], api_key="", cache=False, num_retries=0,
                   temperature=cfg["execution_temperature"], max_tokens=cfg["execution_max_tokens"],
                   seed=cfg["seed"], num_ctx=cfg["ollama_num_ctx"], timeout=cfg["request_timeout"], keep_alive="5m")
    if cfg["ollama_num_gpu"] is not None:
        options["num_gpu"] = cfg["ollama_num_gpu"]
    lm = MeasuredLM("ollama_chat/"+cfg["ollama_model"], **options)
    dspy.configure(lm=lm)
    metadata = {"version":version, "model":model, "details":show.get("details"),
                "model_info":show.get("model_info"), "options":options, "ps":ollama_snapshot(cfg)}
    return lm, metadata

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def rng_state():
    return dict(python=random.getstate(), numpy=np.random.get_state(), torch=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None)

def restore_rng(states):
    random.setstate(states["python"]); np.random.set_state(states["numpy"]); torch.set_rng_state(states["torch"])
    if states["cuda"] is not None:
        if not torch.cuda.is_available():
            raise RuntimeError("Checkpoint has CUDA RNG state; resume in its original GPU environment")
        torch.cuda.set_rng_state_all(states["cuda"])

@contextmanager
def isolated_rng(seed):
    saved = rng_state()
    seed_everything(seed)
    try:
        yield
    finally:
        restore_rng(saved)

def hardware_metadata():
    return dict(python=sys.version, packages=VERSIONS, cuda_available=torch.cuda.is_available(),
                cuda_version=torch.version.cuda,
                gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
                gpu_total_bytes=torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None)

# 5. Dataset Loading
Official [HotpotQA](https://huggingface.co/datasets/hotpotqa/hotpot_qa), `distractor` setting. The dataset commit is resolved and pinned on first load; rerun with an explicit commit to reproduce a historical download.

In [ ]:
def load_hotpotqa(cfg):
    cfg = copy.deepcopy(cfg)
    try:
        if cfg["dataset_revision"] is None:
            cfg["dataset_revision"] = HfApi().dataset_info(cfg["dataset_name"]).sha
        # These are labeled splits. No unlabeled official test is loaded or scored.
        datasets = load_dataset(cfg["dataset_name"], cfg["dataset_config"],
                                revision=cfg["dataset_revision"])
        if not {"train", "validation"}.issubset(datasets):
            raise ValueError("Required labeled HotpotQA splits unavailable")
    except Exception as exc:
        raise RuntimeError("Cannot load official HotpotQA distractor; check network/cache/revision: "+str(exc)) from exc
    return cfg, datasets

def format_hotpot_input(question, context):
    """Use only question and ALL provided paragraphs, in original order."""
    titles, paragraphs = context["title"], context["sentences"]
    if len(titles) != len(paragraphs):
        raise ValueError("Malformed HotpotQA context: mismatched titles/paragraphs")
    sections = [f"Title: {title}\n{''.join(sentences)}" for title, sentences in zip(titles, paragraphs)]
    return "Context:\n"+"\n\n".join(sections)+"\n\nQuestion: "+str(question)

def prepare_hotpot_example(item, split, index):
    if not isinstance(item.get("answer"), str) or not item["answer"].strip():
        raise ValueError("Labeled HotpotQA answer unavailable; refuse fabricated ground truth")
    context = copy.deepcopy(item["context"])
    return dict(id=str(item["id"]), split=split, index=int(index), question=item["question"],
                answer=item["answer"], input=format_hotpot_input(item["question"], context),
                context=context, metadata={key:copy.deepcopy(item.get(key))
                    for key in ("type", "level", "supporting_facts")})

def dataset_example(dataset, split, index):
    return prepare_hotpot_example(dataset[split][int(index)], split, index)

def stable_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def write_json(path, value, overwrite=False):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and not overwrite:
        raise FileExistsError(f"Refusing overwrite: {path}")
    tmp = path.with_name(path.name+"."+uuid.uuid4().hex+".tmp")
    with tmp.open("w",encoding="utf-8") as stream:
        json.dump(value, stream, indent=2, ensure_ascii=False, allow_nan=False)
        stream.flush(); os.fsync(stream.fileno())
    if overwrite:
        os.replace(tmp,path)
    else:
        os.link(tmp,path)
        tmp.unlink()


# 6. Dataset Sampling and Episode Schedule
A dataset/seed/size/revision manifest is shared across K and presets. Training and validation are disjoint subsets of the labeled training split; held-out testing is sampled from the labeled validation split. Episodes draw with replacement from the fixed training pool.


In [ ]:
def selection_payload(cfg, datasets):
    ntrain, ntest = len(datasets["train"]), len(datasets["validation"])
    if cfg["train_pool_size"]+cfg["validation_size"] > ntrain or cfg["test_size"] > ntest:
        raise ValueError("Requested subset exceeds official split; reduce configured sizes")
    permutation = np.random.default_rng(cfg["seed"]).permutation(ntrain)
    train_idx = permutation[:cfg["train_pool_size"]].tolist()
    val_idx = permutation[cfg["train_pool_size"]:cfg["train_pool_size"]+cfg["validation_size"]].tolist()
    test_idx = np.random.default_rng(cfg["seed"]+1).permutation(ntest)[:cfg["test_size"]].tolist()
    subsets = {"train":[dataset_example(datasets,"train",i) for i in train_idx],
               "validation":[dataset_example(datasets,"train",i) for i in val_idx],
               "test":[dataset_example(datasets,"validation",i) for i in test_idx]}
    schedule = np.random.default_rng(cfg["seed"]+2).integers(0,cfg["train_pool_size"],size=cfg["schedule_episodes"]).tolist()
    identity = {key:cfg[key] for key in ["dataset_name","dataset_config","dataset_revision","seed",
                                        "train_pool_size","validation_size","test_size","schedule_episodes"]}
    return dict(identity=identity, indices=dict(train=train_idx,validation=val_idx,test=test_idx),
                ids={name:[item["id"] for item in rows] for name,rows in subsets.items()},
                content_hashes={name:stable_hash(rows) for name,rows in subsets.items()},
                episode_pool_offsets=schedule,
                episode_question_ids=[subsets["train"][offset]["id"] for offset in schedule]), subsets

def prepare_selection(cfg, datasets):
    payload, subsets = selection_payload(cfg,datasets)
    selection_id = stable_hash(payload)
    path = Path(cfg["results_root"])/"hotpotqa"/"selections"/(stable_hash(payload["identity"])+".json")
    if path.exists():
        persisted = json.loads(path.read_text(encoding="utf-8"))
        if persisted != payload:
            raise RuntimeError("Persisted dataset selection/schedule mismatch; do not overwrite it")
    else:
        write_json(path,payload)
    assert set(payload["indices"]["train"]).isdisjoint(payload["indices"]["validation"])
    assert set(payload["ids"]["train"]).isdisjoint(payload["ids"]["test"])
    assert set(payload["ids"]["validation"]).isdisjoint(payload["ids"]["test"])
    return dict(subsets=subsets, selection=payload, selection_id=selection_id, selection_path=str(path),
                full_test=datasets["validation"])

# Repeated draws are allowed, not forced. Small schedules may happen to contain no duplicate.
# Schedule identity never depends on K or preset; episodes use its leading prefix.

# 7. DSPy Action Space and Signatures
The released module pool and all fifteen signatures are retained, including signatures that do not naturally fit math QA. The policy selects one module, one signature, then deterministic `stop`; this released toy implementation therefore executes one DSPy stage per pipeline. No ReAct, retrieval, teleprompter, or new action head.

In [ ]:
MODULES = ["CoT", "Predict"]
SIGNATURES = ["question -> answer", "text -> summary", "question -> reasoning", "question -> hypothesis",
              "problem -> solution", "problem_description -> explanation", "context -> summary", "context -> briefing",
              "word_problem -> solution", "math_problem -> answer", "prompt -> response", "query -> response",
              "text -> response", "prompt -> generated_text", "query -> generated_text"]
ACTIONS = MODULES + SIGNATURES + ["stop"]

def valid_actions(partial):
    return MODULES if not partial else SIGNATURES if len(partial)==1 else ["stop"]

# 8. GPT-2 Policy Network Initialization
Each new run loads the same immutable pretrained GPT-2 revision and a fresh Adam optimizer; no checkpoint is searched for implicitly. Evaluation mode disables the pretrained dropout, matching the released notebook; autograd remains enabled during updates. Only one policy is held by the experiment session. Release an old session before starting another K.

Adam uses `foreach=False` to reduce temporary optimizer memory while retaining its mathematical update and default betas/epsilon. Initial parameter hashes are persisted; cross-K warm starts are prohibited.

In [ ]:
def parameter_digest(model):
    digest = hashlib.sha256()
    for name, parameter in model.named_parameters():
        digest.update(name.encode()); digest.update(parameter.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()

def initialize_policy(cfg):
    if cfg["policy_device"] == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA policy requested but unavailable. Select the GPU kernel or explicitly use CPU")
    seed_everything(cfg["seed"])
    try:
        tokenizer = GPT2Tokenizer.from_pretrained(cfg["policy_model"],revision=cfg["policy_revision"])
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.truncation_side = "left"  # preserves Pipeline suffix if a custom long prompt exceeds 1024
        model = GPT2LMHeadModel.from_pretrained(cfg["policy_model"],revision=cfg["policy_revision"])
    except Exception as exc:
        raise RuntimeError("GPT-2 files unavailable: "+str(exc)) from exc
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False
    model.to(cfg["policy_device"]).eval()
    optimizer = torch.optim.Adam(model.parameters(),lr=cfg["learning_rate"],foreach=False)
    if cfg.get("policy_sampling") != "unique-signature-token-label-v1":
        raise ValueError("This notebook requires the unique signature scorer; old checkpoints are incompatible")
    # Unique existing GPT-2 label tokens distinguish signature scores.
    if len(SIGNATURES)>26 or len(set(SIGNATURES))!=len(SIGNATURES):
        raise ValueError("Signature list must be unique and contain at most 26 entries")
    signature_tokens={}
    for i, signature in enumerate(SIGNATURES):
        label=" "+chr(ord("A")+i)
        ids=tokenizer.encode(label,add_special_tokens=False)
        if len(ids)!=1:
            raise ValueError("Each signature label must encode as exactly one GPT-2 token")
        signature_tokens[signature]=ids[0]
    if len(set(signature_tokens.values()))!=len(signature_tokens):
        raise ValueError("Signature scoring labels must have distinct token IDs")
    first_tokens={action:tokenizer.encode(action,add_special_tokens=False)[0] for action in ACTIONS}
    first_tokens.update(signature_tokens)
    cfg["signature_action_token_ids"]=dict(signature_tokens)
    initial_digest = parameter_digest(model)
    return model, tokenizer, optimizer, first_tokens, initial_digest

def release_experiment(session):
    if session is not None:
        for key in ("policy","optimizer","tokenizer"):
            session.pop(key,None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# 9. Policy Action Generation & Analytical Pre-Allocation
Scores are the first GPT-2 token logits of valid action **entries**. Fifteen signatures share nine first-token IDs: collision entries remain separate and cannot learn relative preference within a collision group. `log_softmax` preserves the sampled categorical probabilities; no full-string scoring is introduced. The same prompt and partial-action state are used for rollout and differentiable replay.

**Pillar 1 (Layer B — Analytical Pre-Allocation):**
GPT-2 computes exact categorical probabilities $\pi_j$ over all 30 discrete pipelines ($2\text{ modules} \times 15\text{ signatures}$) using the chain rule:
$$P(\text{Pipeline}_{m,s}) = P(\text{Module } m \mid \text{Prompt}) \times P(\text{Signature } s \mid \text{Prompt}, m)$$
From $\pi_j$, the controller computes the Expected Unique Coverage:
$$\mathbb{E}[U(K)] = \sum_{j=1}^{30} \Big[ 1 - (1 - \pi_j)^K \Big] \quad \text{for } K \in \{5, 8, 10, 15\}$$
and maximizes penalized utility $\mathcal{J}(K) = \mathbb{E}[U(K)] - \lambda K$ (with $\lambda = 0.25$) to set $K_{\text{ceiling}} \in \{5, 8, 10, 15\}$. If the policy has collapsed onto 1–2 actions, $\Delta U < \lambda \Delta K$, capping $K_{\text{ceiling}} = 5$. Prompts consistently unsolvable (historical pass rate $\hat{p}_x \approx 0$) are regularized to $K_{\text{ceiling}} = 5$.


In [ ]:
def action_log_probs(model, tokenizer, first_tokens, prompt, partial, cfg):
    choices = valid_actions(partial)
    text = f"Prompt: {prompt} Pipeline: {' '.join(partial)}"
    encoded = tokenizer(text,return_tensors="pt",padding=True,truncation=True,
                        max_length=min(cfg["max_policy_tokens"],model.config.n_positions))
    device = next(model.parameters()).device
    encoded = {key:value.to(device) for key,value in encoded.items()}
    logits = model(**encoded,use_cache=False).logits[0,-1]
    ids = torch.tensor([first_tokens[action] for action in choices],device=device)
    return choices, torch.log_softmax(logits[ids],dim=0)

def sample_actions(model, tokenizer, first_tokens, prompt, cfg):
    partial, log_probs = [], []
    with torch.no_grad():
        for _ in range(cfg["max_pipeline_steps"]):
            choices, scores = action_log_probs(model,tokenizer,first_tokens,prompt,partial,cfg)
            index = int(torch.multinomial(scores.exp(),1).item())
            action = choices[index]
            partial.append(action); log_probs.append(float(scores[index]))
            if action == "stop":
                break
    return dict(actions=partial,log_probs=log_probs)

def replay_log_probs(model, tokenizer, first_tokens, prompt, trajectory, cfg):
    partial, log_probs = [], []
    for action in trajectory["actions"]:
        choices, scores = action_log_probs(model,tokenizer,first_tokens,prompt,partial,cfg)
        if action not in choices:
            raise ValueError("Trajectory action violates released action grammar")
        log_probs.append(scores[choices.index(action)])
        partial.append(action)
    return log_probs

def action_collision_report(first_tokens):
    groups = {}
    for signature in SIGNATURES:
        groups.setdefault(first_tokens[signature],[]).append(signature)
    return {token:actions for token,actions in groups.items() if len(actions)>1}

def compute_pipeline_distribution(model, tokenizer, first_tokens, prompt, cfg):
    """
    Pillar 1: Computes exact categorical probabilities pi_j over all 30 (Module x Signature) pipelines
    using the chain rule: P(Pipeline_{m,s}) = P(Module m | Prompt) * P(Signature s | Prompt, m).
    """
    with torch.no_grad():
        _, mod_scores = action_log_probs(model, tokenizer, first_tokens, prompt, [], cfg)
        mod_probs = mod_scores.exp().cpu().numpy()
        
        pi = {}
        for m_idx, m in enumerate(MODULES):
            _, sig_scores = action_log_probs(model, tokenizer, first_tokens, prompt, [m], cfg)
            sig_probs = sig_scores.exp().cpu().numpy()
            for s_idx, s in enumerate(SIGNATURES):
                pi[(m, s)] = float(mod_probs[m_idx] * sig_probs[s_idx])
                
        prob_array = np.array(list(pi.values()), dtype=np.float64)
        prob_array = prob_array / prob_array.sum()
        return pi, prob_array

def solve_preallocation_ceiling(prob_array, cfg, historical_pass_rate=None):
    """
    Pillar 1: Analytical Pre-Allocation Controller.
    Solves max_K [ E[U(K)] - lambda * K ] over K in {5, 8, 10, 15} with lambda = 0.25.
    Regularizes hopeless prompts (historical pass rate == 0 after >= 2 attempts) to K=5.
    """
    levels = cfg.get("adaptive_levels", [5, 8, 10, 15])
    lambda_penalty = cfg.get("lambda_penalty", 0.25)
    
    expected_coverage = {}
    utility = {}
    for k in levels:
        eu_k = float(np.sum(1.0 - (1.0 - prob_array) ** k))
        expected_coverage[k] = eu_k
        utility[k] = eu_k - lambda_penalty * k
    
    entropy = -float(np.sum(prob_array * np.log(prob_array + 1e-12)))
    
    # GVM-RAFT Difficulty Regularization
    if historical_pass_rate is not None:
        p_hat, attempts = historical_pass_rate
        if attempts >= 2 and p_hat == 0.0:
            return levels[0], expected_coverage, utility, entropy
            
    # Staged marginal efficiency check: upgrade K_a -> K_b iff delta_U >= lambda * delta_K
    chosen_k = levels[0]
    for i in range(len(levels) - 1):
        k_curr = levels[i]
        k_next = levels[i + 1]
        delta_k = k_next - k_curr
        delta_u = expected_coverage[k_next] - expected_coverage[k_curr]
        if delta_u >= lambda_penalty * delta_k:
            chosen_k = k_next
        else:
            break
            
    return chosen_k, expected_coverage, utility, entropy


# 10. DSPy Pipeline Generation
`generate_pipeline()` always receives the intended policy explicitly. This corrects the original evaluator's accidental use of a global policy. Generated actions and signatures are logged for every attempted candidate.

In [ ]:
def generate_pipeline(model, tokenizer, first_tokens, prompt, cfg):
    trajectory = sample_actions(model,tokenizer,first_tokens,prompt,cfg)
    actions = trajectory["actions"]
    trajectory["module"] = actions[0] if actions else None
    trajectory["signature"] = actions[1] if len(actions)>1 else None
    return trajectory

# 11. DSPy Pipeline Execution & Deduplication Caching
Candidates execute sequentially. The original square-bracket parser and standard DSPy LM invocation are unchanged.

**Pillar 2 (Layer A — Intra-Episode Deduplication & Deterministic Caching):**
AutoDSPy candidate execution at `temperature = 0.0` is deterministic. Inside each episode, an intra-episode cache `(module, signature) -> (result, reward)` eliminates duplicate calls to Ollama. When duplicate pipelines are drawn within a group, Llama-3.1 inference runs only once per unique pipeline; duplicate candidates reuse cached results without incurring redundant GPU compute, while preserving exact group statistics ($K$) for GRPO advantage computation.


In [ ]:
def execute_pipeline(trajectory, prompt, lm):
    started = time.perf_counter()
    try:
        module, signature = trajectory["module"],trajectory["signature"]
        if module not in MODULES or signature not in SIGNATURES:
            raise ValueError("Invalid pipeline module or signature")
        input_field, output_field = [part.strip() for part in signature.split("->")]
        program = dspy.ChainOfThought(signature) if module == "CoT" else dspy.Predict(signature)
        instruction = "system instruction: Must give your final answer in square brackets without fail! e.g., [final answer] like this: [5] \n prompt: "
        with dspy.context(lm=lm):
            response = program(**{input_field:instruction+prompt})
        output = response.get(output_field)
        if output is None:
            raise ValueError("DSPy returned no signature output")
        return dict(output=str(output),success=True,error=None,seconds=time.perf_counter()-started)
    except Exception as exc:
        return dict(output=None,success=False,error=f"{type(exc).__name__}: {exc}",seconds=time.perf_counter()-started)

# 12. Reward Calculation
HotpotQA answer exact match normalizes casing, punctuation, articles, and whitespace. In `original_semantic` (default), when exact match fails, a semantic judge LLM call evaluates answer equivalence in $[0.0, 1.0]$. `answer_exact` disables the semantic fallback and scores only normalized exact string matches as $1.0$ or $0.0$.

In [ ]:
import re
import string
import collections

def normalize_answer(text):
    """Official HotpotQA answer normalization: case, ASCII punctuation, articles, whitespace."""
    value = str(text).lower()
    value = ''.join(character for character in value if character not in string.punctuation)
    value = re.sub(r"\b(a|an|the)\b", " ", value)
    return " ".join(value.split())

def extract_answer(response):
    if response is None:
        return None
    text = str(response).strip()
    brackets = re.findall(r"\[([^\[\]]+)\]", text)
    return brackets[-1].strip() if brackets else text

def compare_answers(response, target):
    prediction = extract_answer(response)
    return prediction is not None and normalize_answer(prediction) == normalize_answer(target)

def answer_f1(prediction, target):
    """Official evaluator's answer-only F1, precision and recall; not a training reward."""
    if prediction is None:
        return 0.0, 0.0, 0.0
    prediction, target = normalize_answer(prediction), normalize_answer(target)
    pred_tokens, target_tokens = prediction.split(), target.split()
    common = collections.Counter(pred_tokens) & collections.Counter(target_tokens)
    num_same = sum(common.values())
    if len(pred_tokens) == 0 or len(target_tokens) == 0:
        same = float(prediction == target)
        return same, same, same
    if num_same == 0:
        return 0.0, 0.0, 0.0
    precision = 1.0 * num_same / len(pred_tokens)
    recall = 1.0 * num_same / len(target_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return f1, precision, recall

def extract_text_answer(response):
    """Explicit final text, or a plain single-line answer; no arbitrary reasoning suffix."""
    if response is None:
        return None
    text = str(response).strip()
    if not text:
        return None
    marked = list(re.finditer(r'(?:^|\n)\s*(?:the\s+)?(?:final\s+answer|answer|final\s+result)\s*(?:is\b|:|=)\s*', text, re.I))
    if '####' in text:
        text = text.rsplit('####', 1)[1].strip()
    elif marked:
        text = text[marked[-1].end():].strip()
    elif re.search(r'\[[^\[\]]+\]\s*[.!]?\s*$', text):
        match = re.search(r'\[([^\[\]]+)\]\s*[.!]?\s*$', text)
        prefix = text[:match.start()]
        if '[' in prefix or re.search(r'\b(?:intermediate|partial|previous)\b', prefix, re.I):
            return None
        text = match.group(1).strip()
    elif '\n' in text:
        return None
    text = text.strip()
    for _ in range(3):
        before = text
        for left, right in [('**','**'), ('`','`'), ('[',']')]:
            if text.startswith(left) and text.endswith(right) and len(text) > len(left)+len(right):
                text = text[len(left):-len(right)].strip()
        if text == before:
            break
    return text if text and '\n' not in text and normalize_answer(text) else None

def reward_prediction(response, cfg=None):
    mode = (cfg or {}).get("reward_mode", "original_semantic")
    return extract_text_answer(response) if mode == "answer_exact" else extract_answer(response)

def parse_judge_score(text):
    value = str(text).strip()
    bracket = re.fullmatch(r"\s*\[\s*([^\[\]]+)\s*\]\s*", value)
    if bracket:
        value = bracket.group(1).strip()
    if re.fullmatch(r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)", value) is None:
        raise ValueError("Judge did not return a single finite numeric score")
    score = float(value)
    if not math.isfinite(score):
        raise ValueError("Nonfinite judge score")
    return min(1.0, max(0.0, score))

def compute_reward(prompt, response, target, lm, cfg=None, reward_context=None, identifiers=None):
    mode = (cfg or {}).get("reward_mode", "original_semantic")
    if mode not in ("original_semantic", "answer_exact"):
        raise ValueError("Unknown reward mode")
    prediction = reward_prediction(response, cfg)
    exact = prediction is not None and normalize_answer(prediction) == normalize_answer(target)
    f1, prec, rec = answer_f1(prediction, target)
    if response is None:
        return dict(reward=0.0, exact_match=False, answer_f1=0.0, semantic_used=False, judge_score=None, judge_error=None)
    if exact:
        return dict(reward=1.0, exact_match=True, answer_f1=1.0, semantic_used=False, judge_score=None, judge_error=None)
    if mode == "answer_exact":
        return dict(reward=0.0, exact_match=False, answer_f1=f1, semantic_used=False, judge_score=None, judge_error=None)
    judge_prompt = (
        "Evaluate whether the following response correctly answers the prompt based on the ground truth and give your final score in the square brackets[final score]. only the value in the [] should in your response and nothing else.\n"
        f"Prompt: {prompt}\nResponse: {response}\nGround Truth: {target}\n"
        "Return a score in range between 1.0 to 0.0 if the response is correct or partially correct (matches or is equivalent to the ground truth), or 0.0 if incorrect.\nfinal score:[]"
    )
    try:
        outputs = lm(judge_prompt)
        score = parse_judge_score(outputs[0])
        return dict(reward=score, exact_match=False, answer_f1=f1, semantic_used=True, judge_score=score, judge_error=None)
    except Exception as exc:
        return dict(reward=0.0, exact_match=False, answer_f1=f1, semantic_used=True, judge_score=None,
                    judge_error=f"{type(exc).__name__}: {exc}")


# 13. Figure 2 GRPO Loss (1/K Normalized) & Zero-Gradient Masking
$$\bar r = \frac{1}{K}\sum_{i=1}^K r_i,\quad A_i = r_i - \bar r,\quad \mathcal L = -\frac{1}{K}\sum_{i=1}^K\sum_t\log\pi_\theta(a_{i,t}\mid s_{i,t})A_i$$

**Critical Enhancements in v2:**
1. **1/K Loss Normalization:** All Figure 2 losses and backpropagated gradients are normalized by $\frac{1}{K}$, where $K = \text{len}(\text{trajectories})$ is the realized group size ($K \in \{5, 8, 10, 15\}$). This guarantees invariant gradient scale regardless of whether an episode ran at $K=5, 8, 10,$ or $15$.
2. **Pillar 4 (Layer D — Zero-Gradient Optimizer Masking from DAPO):** When all candidates in a group receive identical reward (all-pass or all-fail), centered advantages are all zero ($A_i = 0$). Calling `optimizer.step()` on zero gradients allows Adam momentum buffers (`exp_avg`) to drift arbitrarily. Under Pillar 4, if $\text{all}(A_i == 0)$, we skip `loss.backward()` and `optimizer.step()`, clearing gradients with `optimizer.zero_grad(set_to_none=True)` and logging `optimizer_step = False`.


In [ ]:
def group_advantages(rewards):
    values = np.asarray(rewards, dtype=np.float64)
    if len(values) == 0 or not np.isfinite(values).all():
        raise ValueError("GRPO requires a nonempty finite group")
    return float(values.mean()), values - values.mean()

def figure2_loss(log_prob_trajectories, advantages, group_size=None):
    if len(log_prob_trajectories) != len(advantages):
        raise ValueError("One advantage per candidate is required")
    K = float(group_size if group_size is not None else len(log_prob_trajectories))
    if K <= 0:
        raise ValueError("Group size must be positive")
    # Exact 1/K normalization
    return -(1.0 / K) * sum(sum(log_probs) * float(advantage) for log_probs, advantage in zip(log_prob_trajectories, advantages))

def update_policy(session, prompt, trajectories, rewards):
    if session.get("evaluation_only", False):
        raise RuntimeError("Evaluation-only checkpoint: use explicit resume before updating")
    model, optimizer = session["policy"], session["optimizer"]
    cfg = session["config"]
    K = len(trajectories)
    if K not in cfg["adaptive_levels"] or len(rewards) != K:
        raise ValueError("Episode must contain exactly K candidate rewards")
    mean, advantages = group_advantages(rewards)
    
    # Pillar 4: Zero-Gradient Optimizer Masking (Layer D - from DAPO)
    is_zero_variance = bool(np.all(advantages == 0))
    if is_zero_variance:
        optimizer.zero_grad(set_to_none=True)
        return dict(
            group_mean=mean,
            advantages=advantages.tolist(),
            loss=0.0,
            gradient_norm=0.0,
            zero_advantage_group=True,
            optimizer_step=False,
            skip_reason="zero_variance_group",
        )
    
    optimizer.zero_grad(set_to_none=True)
    total_loss = 0.0
    model.eval()
    for trajectory, advantage in zip(trajectories, advantages):
        scores = replay_log_probs(model, session["tokenizer"], session["first_tokens"], prompt, trajectory, cfg)
        replay_values = [float(score.detach()) for score in scores]
        if not np.allclose(replay_values, trajectory["log_probs"], rtol=2e-5, atol=2e-5):
            optimizer.zero_grad(set_to_none=True)
            raise RuntimeError("Replay differs from rollout; parameters/mode/state changed before update")
        # 1/K normalized candidate loss
        loss = figure2_loss([scores], [advantage], group_size=K)
        if not torch.isfinite(loss):
            optimizer.zero_grad(set_to_none=True)
            raise RuntimeError("Nonfinite Figure 2 loss; no optimizer step permitted")
        total_loss += float(loss.detach())
        loss.backward()
        del loss, scores

    norm_sq = sum(float(parameter.grad.detach().double().square().sum())
                  for parameter in model.parameters() if parameter.grad is not None)
    norm = math.sqrt(norm_sq)
    if not math.isfinite(norm):
        optimizer.zero_grad(set_to_none=True)
        raise RuntimeError("Nonfinite gradient; no optimizer step permitted")
    optimizer.step()
    session.pop("loaded_checkpoint", None)
    optimizer.zero_grad(set_to_none=True)
    return dict(group_mean=mean, advantages=advantages.tolist(), loss=total_loss, gradient_norm=norm,
                zero_advantage_group=False, optimizer_step=True, skip_reason=None)


# 14. GRPO Training Loop (APEX-GRPO 4-Pillar Pipeline)
Every episode uses one scheduled question and attempts candidates sequentially following the **APEX-GRPO waterfall**:
1. **Pre-Allocation (Pillar 1):** Compute exact policy coverage over 30 pipelines and set $K_{\text{ceiling}} \in \{5, 8, 10, 15\}$.
2. **Deduplication Caching (Pillar 2):** Intra-episode cache ensures Llama-3.1 inference runs only once per unique pipeline.
3. **Staged Waterfall Exit (Pillar 3):** Evaluate candidates in stages ($5 \to 8 \to 10 \to 15$); exit immediately on reward contrast or all-correct; rescue only on all-failures if $K_{\text{ceiling}}$ permits.
4. **Masked 1/K Update (Pillar 4):** Mask optimizer steps on zero-variance groups and scale loss by $1/K$.

Periodic checkpoints, validation checkpoints, and safe between-episode interruption preserve completed work.


In [ ]:
def initialize_experiment(cfg, create_run=True):
    cfg, datasets = load_hotpotqa(cfg)
    data = prepare_selection(cfg, datasets)
    lm, llm_metadata = initialize_ollama(cfg)
    model, tokenizer, optimizer, first_tokens, digest = initialize_policy(cfg)
    session = dict(config=cfg, data=data, lm=lm, llm_metadata=llm_metadata, policy=model, tokenizer=tokenizer,
                   optimizer=optimizer, first_tokens=first_tokens, initial_policy_digest=digest,
                   state=dict(episode=0, updates=0, train=[], validation=[], tests=[], best=None, seen=[],
                              cumulative_training_seconds=0.0), run_dir=None, in_episode=False)
    if create_run:
        run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "-" + uuid.uuid4().hex[:8]
        root = Path(cfg["results_root"]) / "hotpotqa" / f"adaptive-{cfg['rescue_mode']}" / f"seed{cfg['seed']}" / cfg["preset"] / run_id
        root.mkdir(parents=True, exist_ok=False); (root / "checkpoints").mkdir()
        session["run_dir"] = root
        write_json(root / "config.json", dict(config=cfg, selection_id=data["selection_id"],
                   selection_path=data["selection_path"], initial_policy_digest=digest,
                   hardware=hardware_metadata(), execution_model=llm_metadata))
    return session

def append_journal(session, event):
    if session["run_dir"] is None:
        return
    with (session["run_dir"] / "attempts.jsonl").open("a", encoding="utf-8") as stream:
        stream.write(json.dumps(event, allow_nan=False) + "\n"); stream.flush(); os.fsync(stream.fileno())

def reserved_training_candidates(session):
    journal = session["run_dir"] / "attempts.jsonl"
    if not journal.exists():
        return 0
    return sum(row.get("kind") == "training_reservation" for row in read_jsonl(journal))

def train_grpo(session):
    if session.get("evaluation_only", False):
        raise RuntimeError("Evaluation-only session: initialize a new run or use explicit resume")
    cfg, state = session["config"], session["state"]
    validated = {row["episode"] for row in state["validation"]}
    if state["episode"] == cfg["num_episodes"] and (cfg["num_episodes"] in validated or state.get("budget_exhausted", False)):
        raise RuntimeError("Run completed; create a new run or evaluate its checkpoint")
    pending_validation = (state.get("budget_exhausted", False) or state["episode"] == 0 or state["episode"] % cfg["validation_interval"] == 0 or state["episode"] == cfg["num_episodes"]) and state["episode"] not in validated
    if pending_validation:
        validation = evaluate_module(session, "validation", episode=state["episode"])
        state["validation"].append(validation); save_checkpoint(session, validation=validation)
    elif not (session["run_dir"] / "latest.json").exists():
        save_checkpoint(session)
    if state["episode"] == cfg["num_episodes"] or state.get("budget_exhausted", False):
        write_summary(session)
        return session
    try:
        while state["episode"] < cfg["num_episodes"]:
            remaining = None if cfg["candidate_budget"] is None else cfg["candidate_budget"] - reserved_training_candidates(session)
            if remaining is not None and remaining < cfg["adaptive_levels"][0]:
                state["budget_exhausted"] = True
                append_journal(session, dict(kind="budget_exhausted", remaining=remaining, episode=state["episode"]))
                validation = None
                if not any(row["episode"] == state["episode"] for row in state["validation"]):
                    validation = evaluate_module(session, "validation", episode=state["episode"])
                    state["validation"].append(validation)
                save_checkpoint(session, validation=validation); write_summary(session)
                break
            episode = state["episode"] + 1
            offset = session["data"]["selection"]["episode_pool_offsets"][episode - 1]
            example = session["data"]["subsets"]["train"][offset]
            prompt = example["input"]  # Formatted context + question for HotpotQA
            session["in_episode"] = True
            if next(session["policy"].parameters()).is_cuda:
                torch.cuda.reset_peak_memory_stats()
            begin = time.perf_counter(); call_start = len(session["lm"].events)

            # -------------------------------------------------------------
            # Pillar 1: Analytical Pre-Allocation (Layer B)
            # -------------------------------------------------------------
            pi_dict, prob_array = compute_pipeline_distribution(
                session["policy"], session["tokenizer"], session["first_tokens"], prompt, cfg
            )
            q_history = session["state"].setdefault("question_history", {}).get(example["id"])
            k_ceiling, expected_coverage, utility_dict, policy_entropy = solve_preallocation_ceiling(
                prob_array, cfg, historical_pass_rate=q_history
            )

            # -------------------------------------------------------------
            # Pillar 2: Intra-Episode Deduplication Cache (Layer A)
            # & Pillar 3: Deterministic Staged Waterfall Rollout (Layer C)
            # -------------------------------------------------------------
            episode_cache = {}  # (module, signature) -> (result, reward)
            trajectories, results, reward_details = [], [], []
            stop_reason = None
            physical_calls_start = len(session["lm"].events)
            levels = cfg["adaptive_levels"]

            for stage_idx, target_k in enumerate(levels):
                needed = target_k - len(trajectories)
                for _ in range(needed):
                    candidate_idx = len(trajectories)
                    append_journal(session, dict(kind="training_reservation", episode=episode, candidate=candidate_idx))
                    candidate_call_start = len(session["lm"].events)
                    
                    trajectory = generate_pipeline(session["policy"], session["tokenizer"], session["first_tokens"], prompt, cfg)
                    pipeline_key = (trajectory["module"], trajectory["signature"])

                    if pipeline_key in episode_cache:
                        # Intra-episode cache hit: deterministic deduplication
                        cached_result, cached_reward = episode_cache[pipeline_key]
                        result = dict(cached_result)
                        result["cached"] = True
                        reward = dict(cached_reward)
                        reward["cached"] = True
                    else:
                        # Physical execution via Ollama
                        result = execute_pipeline(trajectory, prompt, session["lm"])
                        result["cached"] = False
                        reward = compute_reward(prompt, result["output"], example["answer"], session["lm"], cfg)
                        reward["cached"] = False
                        episode_cache[pipeline_key] = (result, reward)

                    trajectories.append(trajectory); results.append(result); reward_details.append(reward)
                    append_journal(session, dict(kind="training_attempt", episode=episode, candidate=candidate_idx,
                                                 question_id=example["id"], execution=result, reward=reward,
                                                 cached=result.get("cached", False),
                                                 resources=call_statistics(session["lm"].events[candidate_call_start:]),
                                                 timestamp=datetime.now(timezone.utc).isoformat()))

                    if cfg["candidate_budget"] is not None and reserved_training_candidates(session) >= cfg["candidate_budget"]:
                        stop_reason = "budget_exhausted"
                        break

                if stop_reason is not None:
                    break

                # Staged evaluation at level boundary
                failures = any(not item["success"] for item in results) or any(item.get("judge_error") for item in reward_details)
                current_rewards = [item["reward"] for item in reward_details]
                decision = adaptive_decision(current_rewards, failures, cfg, k_ceiling=k_ceiling)
                if decision is not None:
                    stop_reason = decision
                    break

            rewards = [detail["reward"] for detail in reward_details]
            realized_k = len(rewards)

            # Update historical pass rate for GVM-RAFT difficulty tracking
            mean_r = float(np.mean(rewards)) if rewards else 0.0
            prev_pass, prev_count = session["state"].setdefault("question_history", {}).get(example["id"], (0.0, 0))
            new_count = prev_count + 1
            new_pass = ((prev_pass * prev_count) + (1.0 if mean_r > 0 else 0.0)) / new_count
            session["state"]["question_history"][example["id"]] = (new_pass, new_count)

            # -------------------------------------------------------------
            # Pillar 4: Zero-Gradient Masking & 1/K Policy Update (Layer D)
            # -------------------------------------------------------------
            update = update_policy(session, prompt, trajectories, rewards)

            seconds = time.perf_counter() - begin
            seen = set(state["seen"]); seen.add(example["id"])
            calls = call_statistics(session["lm"].events[call_start:])
            physical_lm_calls = len(session["lm"].events[physical_calls_start:])
            cached_calls = sum(r.get("cached", False) for r in results)
            unique_pipelines = len(set(tuple(t["actions"][:2]) for t in trajectories))

            row = dict(
                dataset=cfg["dataset"],
                k=realized_k,
                k_ceiling=k_ceiling,
                expected_coverage=expected_coverage,
                policy_entropy=policy_entropy,
                unique_pipelines_count=unique_pipelines,
                unique_pipeline_ratio=float(unique_pipelines / realized_k) if realized_k > 0 else 0.0,
                physical_lm_calls=physical_lm_calls,
                cached_pipeline_executions=cached_calls,
                stop_reason=stop_reason,
                allocation_version=cfg["allocation_version"],
                seed=cfg["seed"],
                episode=episode,
                question_id=example["id"],
                question_pool_offset=offset,
                actions=[t["actions"] for t in trajectories],
                signatures=[t["signature"] for t in trajectories],
                rewards=rewards,
                mean_group_reward=update["group_mean"],
                reward_std=float(np.std(rewards)),
                reward_variance=float(np.var(rewards)),
                advantages=update["advantages"],
                zero_advantage_group=update["zero_advantage_group"],
                optimizer_step=update["optimizer_step"],
                skip_reason=update.get("skip_reason"),
                grpo_loss=update["loss"],
                gradient_norm=update["gradient_norm"],
                optimizer_update_count=state["updates"] + int(update["optimizer_step"]),
                unique_training_questions=len(seen),
                attempted_pipeline_evaluations=realized_k,
                successful_pipeline_evaluations=sum(result["success"] for result in results),
                failed_pipeline_evaluations=sum(not result["success"] for result in results),
                episode_seconds=seconds,
                cumulative_training_seconds=state["cumulative_training_seconds"] + seconds,
                average_pipeline_execution_latency=float(np.mean([result["seconds"] for result in results])),
                pipeline_execution_seconds=sum(result["seconds"] for result in results),
                candidate_details=[dict(trajectory=t, execution=r, reward=w) for t, r, w in zip(trajectories, results, reward_details)],
                out_of_memory_failures=sum("out of memory" in (result["error"] or "").lower() for result in results),
                execution_failures=[result["error"] for result in results if result["error"]],
                judge_failures=[detail["judge_error"] for detail in reward_details if detail["judge_error"]],
                ollama_memory=ollama_snapshot(cfg),
                **calls,
                **gpu_statistics(session["policy"])
            )
            state["train"].append(row)
            state["episode"] = episode
            if update["optimizer_step"]:
                state["updates"] += 1
            state["seen"] = sorted(seen)
            state["cumulative_training_seconds"] += seconds
            session["in_episode"] = False
            synchronize_logs(session)

            validation = None
            if episode % cfg["validation_interval"] == 0 or episode == cfg["num_episodes"]:
                validation = evaluate_module(session, "validation", episode=episode)
                state["validation"].append(validation)
                print(f"--> [Validation Ep {episode}] EM: {validation['exact_match_accuracy']*100:.1f}% | "
                      f"F1: {validation['answer_f1']:.3f} | Mean Reward (Semantic): {validation['mean_reward']:.3f} ({validation['mean_reward']*100:.1f}%)")
            if validation is not None or episode % cfg["checkpoint_interval"] == 0 or episode == cfg["num_episodes"]:
                save_checkpoint(session, validation=validation)
            write_summary(session)
            print(f"K={realized_k} (ceil={k_ceiling}, uniq={unique_pipelines}) episode={episode}/{cfg['num_episodes']} "
                  f"reward={row['mean_group_reward']:.3f} loss={row['grpo_loss']:.4f} grad={row['gradient_norm']:.4f} "
                  f"opt_step={row['optimizer_step']}")
    except BaseException as exc:
        append_journal(session, dict(kind="interruption", episode=state["episode"] + int(session["in_episode"]),
                       error=f"{type(exc).__name__}: {exc}", in_episode=session["in_episode"],
                       out_of_memory="out of memory" in str(exc).lower(),
                       gpu_memory=gpu_statistics(session["policy"]),
                       timestamp=datetime.now(timezone.utc).isoformat()))
        if not session["in_episode"]:
            save_checkpoint(session)
            write_summary(session)
        else:
            print("Episode interrupted. Resume the latest.json checkpoint; incomplete work was not committed.")
        raise
    return session


# 15. Checkpoint Management
Immutable filenames prevent accidental checkpoint overwrite; atomic latest/best pointers stay in the explicit run. Select best by highest validation numeric EM; **earliest episode wins ties**. Every validation state, including episode 0, is checkpointed. Resume only the latest durable checkpoint; historical checkpoints are available for evaluation.

Checkpoints contain policy, Adam, complete config, all RNG states, selection/schedule, metrics and execution identity. Load only your own trusted checkpoints (`weights_only=False` permits Python/NumPy RNG restoration). Final-test results never select checkpoints.

In [ ]:
def checkpoint_payload(session):
    if session.get("evaluation_only", False):
        raise RuntimeError("Do not save training checkpoints from an evaluation-only session")
    if session["in_episode"]:
        raise RuntimeError("Cannot checkpoint an incomplete episode")
    return dict(format_version=1, config=copy.deepcopy(session["config"]),
                policy=session["policy"].state_dict(), optimizer=session["optimizer"].state_dict(),
                state=copy.deepcopy(session["state"]), rng=rng_state(),
                selection=copy.deepcopy(session["data"]["selection"]), selection_id=session["data"]["selection_id"],
                initial_policy_digest=session["initial_policy_digest"], execution_model=session["llm_metadata"],
                hardware=hardware_metadata(), run_dir=str(session["run_dir"]),
                saved_at=datetime.now(timezone.utc).isoformat())

def atomic_torch_save(path, payload):
    path = Path(path)
    if path.exists():
        raise FileExistsError(f"Refusing checkpoint overwrite: {path}")
    temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    with temporary.open("wb") as stream:
        torch.save(payload, stream); stream.flush(); os.fsync(stream.fileno())
    os.link(temporary, path); temporary.unlink()

def save_checkpoint(session, validation=None):
    state = session["state"]; root = session["run_dir"]
    if root is None:
        raise RuntimeError("No active run directory")
    path = root / "checkpoints" / f"episode_{state['episode']:04d}_{uuid.uuid4().hex[:8]}.pt"
    old_best = state["best"]
    improved = validation is not None and (old_best is None or validation["exact_match_accuracy"] > old_best["score"])
    if improved:
        state["best"] = dict(
            score=validation["exact_match_accuracy"],
            exact_match_accuracy=validation["exact_match_accuracy"],
            answer_f1=validation.get("answer_f1"),
            mean_reward=validation.get("mean_reward"),
            episode=state["episode"],
            checkpoint=str(path)
        )
    try:
        atomic_torch_save(path, checkpoint_payload(session))
    except BaseException:
        state["best"] = old_best
        raise
    write_json(root / "latest.json", dict(checkpoint=str(path), episode=state["episode"]), overwrite=True)
    if improved:
        write_json(root / "best.json", state["best"], overwrite=True)
    synchronize_logs(session)
    backup_results(session["config"])
    return path

def compatible_config(expected, actual):
    ignored = {"results_root"}
    left = {k: v for k, v in expected.items() if k not in ignored}
    right = {k: v for k, v in actual.items() if k not in ignored}
    differing = [key for key in set(left) | set(right) if left.get(key) != right.get(key)]
    if differing:
        raise ValueError("Checkpoint configuration mismatch: " + ", ".join(sorted(differing)))

def load_checkpoint_into(session, path, for_resume=False):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Selected checkpoint not found: {path}")
    payload = torch.load(path, map_location="cpu", weights_only=False)
    if payload.get("format_version") != 1:
        raise ValueError("Unsupported checkpoint format")
    compatible_config(session["config"], payload["config"])
    if payload["selection_id"] != session["data"]["selection_id"] or payload["selection"] != session["data"]["selection"]:
        raise ValueError("Checkpoint dataset/schedule mismatch")
    if payload["initial_policy_digest"] != session["initial_policy_digest"]:
        raise ValueError("Pretrained GPT-2 initial parameters differ")
    old_model = payload["execution_model"].get("model", {})
    new_model = session["llm_metadata"].get("model", {})
    if old_model.get("digest") != new_model.get("digest"):
        raise ValueError("Ollama model digest changed; refuse controlled resume/evaluation")
    if payload["hardware"]["packages"] != VERSIONS or payload["execution_model"].get("version") != session["llm_metadata"].get("version"):
        raise ValueError("Dependency/Ollama versions changed; refuse controlled checkpoint loading")
    root = Path(payload["run_dir"])
    if for_resume:
        if not root.exists():
            raise FileNotFoundError("Original run directory unavailable")
        latest = json.loads((root / "latest.json").read_text(encoding="utf-8"))
        if Path(latest["checkpoint"]).resolve() != path.resolve():
            raise ValueError("Resume only latest durable checkpoint")
        if payload["state"]["episode"] >= payload["config"]["num_episodes"] and any(row["episode"] == payload["config"]["num_episodes"] for row in payload["state"]["validation"]):
            raise ValueError("Run completed; cannot resume as a longer/different preset")
    session["policy"].load_state_dict(payload["policy"]); session["policy"].eval()
    if for_resume:
        session["optimizer"].load_state_dict(payload["optimizer"])
    session["state"] = payload["state"]; session["run_dir"] = root; session["in_episode"] = False
    session["loaded_checkpoint"] = str(path)
    session["evaluation_only"] = not for_resume
    restore_rng(payload["rng"])
    if for_resume:
        synchronize_logs(session)
        if session["state"]["best"] is not None:
            write_json(root / "best.json", session["state"]["best"], overwrite=True)
        write_summary(session)
        append_journal(session, dict(kind="resume", next_episode=session["state"]["episode"] + 1,
                                   checkpoint=str(path), timestamp=datetime.now(timezone.utc).isoformat()))
    del payload; gc.collect()
    return session

def load_selected_experiment(path, cfg, for_resume=False):
    session = initialize_experiment(cfg, create_run=False)
    return load_checkpoint_into(session, path, for_resume=for_resume)


# 16. Validation
One sampled pipeline per fixed validation question, no gradients or optimizer updates. A fixed split-specific policy RNG seed is reused at each checkpoint, then the training RNG is restored. Llama uses fixed seed/temperature; version/device changes can still affect outputs. Strict numeric EM, fractional semantic scores and failures remain distinct.

In [ ]:
def evaluate_module(session, split="validation", episode=None, examples=None):
    if split not in ("validation", "test", "full_test"):
        raise ValueError("Evaluation must use validation/test, never training")
    if examples is None:
        if split == "full_test":
            raise ValueError("Full test requires explicit examples")
        examples = session["data"]["subsets"][split]
    if not examples:
        raise ValueError("Empty evaluation set")
    train_ids = set(session["data"]["selection"]["ids"]["train"])
    if any(example["id"] in train_ids for example in examples):
        raise ValueError("Training examples cannot enter held-out evaluation")
    cfg = session["config"]; model = session["policy"]
    old_mode = model.training; model.eval()
    call_start = len(session["lm"].events); begin = time.perf_counter(); records = []
    seed = cfg["seed"] + cfg["evaluation_seed_offset"] + (0 if split == "validation" else 1)
    try:
        with isolated_rng(seed), torch.no_grad():
            for example in examples:
                question_call_start = len(session["lm"].events)
                trajectory = generate_pipeline(model, session["tokenizer"], session["first_tokens"], example["input"], cfg)
                execution = execute_pipeline(trajectory, example["input"], session["lm"])
                reward = compute_reward(example["input"], execution["output"], example["answer"], session["lm"], cfg)
                append_journal(session, dict(kind="evaluation_attempt", split=split, episode=session["state"]["episode"],
                               question_id=example["id"], execution=execution, reward=reward,
                               resources=call_statistics(session["lm"].events[question_call_start:]),
                               timestamp=datetime.now(timezone.utc).isoformat()))
                records.append(dict(question_id=example["id"], target=example["answer"], prediction=reward_prediction(execution["output"], cfg),
                                    trajectory=trajectory, execution=execution, **reward))
    finally:
        model.train(old_mode)
    exact = sum(record["exact_match"] for record in records)
    f1_mean = float(np.mean([record.get("answer_f1", 0.0) for record in records]))
    scores = [record["judge_score"] for record in records if record["judge_score"] is not None]
    return dict(dataset=cfg["dataset"], k=cfg["k"], seed=cfg["seed"], split=split,
                episode=session["state"]["episode"] if episode is None else episode,
                selected_checkpoint=session.get("loaded_checkpoint"), questions=len(records),
                question_ids=[r["question_id"] for r in records], exact_match_accuracy=exact / len(records),
                answer_f1=f1_mean,
                semantic_fallback_count=sum(record["semantic_used"] for record in records),
                semantic_fallback_mean_score=float(np.mean(scores)) if scores else None,
                semantic_full_credit_count=sum(score == 1.0 for score in scores),
                mean_reward=float(np.mean([r["reward"] for r in records])),
                attempted_pipeline_evaluations=len(records), successful_pipeline_evaluations=sum(r["execution"]["success"] for r in records),
                failed_pipeline_evaluations=sum(not r["execution"]["success"] for r in records),
                cumulative_training_pipeline_evaluations=sum(r["attempted_pipeline_evaluations"] for r in session["state"]["train"]),
                seconds=time.perf_counter() - begin, records=records, **call_statistics(session["lm"].events[call_start:]))

def validate_selected(session):
    if "loaded_checkpoint" not in session:
        raise RuntimeError("Load intended checkpoint before standalone validation")
    result = evaluate_module(session, "validation")
    save_evaluation_artifact(session, result)
    refresh_test_summary(session["run_dir"])
    print(f"Validation EM: {result['exact_match_accuracy']*100:.1f}% | "
          f"F1: {result['answer_f1']:.3f} | Mean Reward (Semantic): {result['mean_reward']:.3f} ({result['mean_reward']*100:.1f}%)")
    return result


# 17. Final Testing
Final testing explicitly loads the validation-selected checkpoint and uses the fixed official-test subset. It never selects checkpoints. Full official-test evaluation is opt-in and can be expensive. Test artifacts do not mutate historical training checkpoints.

In [ ]:
def save_evaluation_artifact(session, result):
    path = session["run_dir"] / f"{result['split']}_{uuid.uuid4().hex[:8]}.json"
    write_json(path, result)
    return path

def test_model(session, full_official=False):
    if "loaded_checkpoint" not in session:
        raise RuntimeError("Final testing requires an explicitly loaded checkpoint")
    training_summary = json.loads((session["run_dir"] / "summary.json").read_text(encoding="utf-8"))
    if not training_summary["completed"]:
        raise RuntimeError("Finish training before final testing")
    best_path = session["run_dir"] / "best.json"
    if not best_path.exists():
        raise RuntimeError("No validation-selected checkpoint")
    best = json.loads(best_path.read_text(encoding="utf-8"))
    if Path(session["loaded_checkpoint"]).resolve() != Path(best["checkpoint"]).resolve():
        raise ValueError("Load validation-selected best checkpoint before final testing")
    if full_official:
        official = session["data"]["full_test"]
        examples = [dict(id=f"hotpotqa:test:{i}", split="test", index=i, question=item["question"], answer=item["answer"], input=format_hotpot_input(item["question"], item["context"]))
                    for i, item in enumerate(official)]
        result = evaluate_module(session, "full_test", examples=examples)
    else:
        result = evaluate_module(session, "test")
    result["checkpoint_selection_rule"] = "maximum validation exact match; earliest episode on ties"
    path = save_evaluation_artifact(session, result)
    refresh_test_summary(session["run_dir"])
    print(f"Held-out EM: {result['exact_match_accuracy']*100:.1f}% | "
          f"F1: {result['answer_f1']:.3f} | Mean Reward (Semantic): {result['mean_reward']:.3f} ({result['mean_reward']*100:.1f}%) | "
          f"artifact: {path}")
    return result


# 18. Metrics and Statistics
JSONL holds episode and validation rows. `attempts.jsonl` records granular attempt-level reservations, LLM call timestamps, physical execution vs deduplication cache hits, and failure recovery. Extended APEX-GRPO metrics track:
- `k_ceiling`: Analytical pre-allocation ceiling computed from Pillar 1.
- `policy_entropy`: Policy action entropy across the 30 pipelines.
- `physical_lm_calls`: Number of physical HTTP calls dispatched to Ollama.
- `cached_pipeline_executions`: Redundant candidate calls avoided via Pillar 2 deduplication.
- `unique_pipelines_count` & `unique_pipeline_ratio`: Policy diversity within the group.
- `optimizer_step` & `skip_reason`: Audit log of Pillar 4 zero-gradient updates vs skips.


In [ ]:
def call_statistics(events):
    def total(key):
        return sum(event[key] for event in events) if all(event.get(key) is not None for event in events) else None
    return dict(execution_llm_calls=len(events), physical_http_requests=None, prompt_tokens=total("prompt_tokens"),
                generated_tokens=total("generated_tokens"), llm_seconds=sum(e["seconds"] for e in events),
                llm_failed_calls=sum(e["error"] is not None for e in events),
                call_count_definition="DSPy LM.forward attempts, including semantic judge and adapter retries")

def gpu_statistics(model):
    if not next(model.parameters()).is_cuda:
        return dict(peak_allocated_gpu_bytes=None, peak_reserved_gpu_bytes=None)
    return dict(peak_allocated_gpu_bytes=torch.cuda.max_memory_allocated(), peak_reserved_gpu_bytes=torch.cuda.max_memory_reserved())

def write_jsonl(path, rows):
    path = Path(path); temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    with temporary.open("w", encoding="utf-8") as stream:
        for row in rows:
            stream.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n")
        stream.flush(); os.fsync(stream.fileno())
    os.replace(temporary, path)

def synchronize_logs(session):
    if session["run_dir"] is not None:
        write_jsonl(session["run_dir"] / "training.jsonl", session["state"]["train"])
        write_jsonl(session["run_dir"] / "validation.jsonl", session["state"]["validation"])

def write_summary(session):
    summary = experiment_summary(session)
    write_json(session["run_dir"] / "summary.json", summary, overwrite=True)
    result = refresh_test_summary(session["run_dir"])
    latest_path = session["run_dir"] / "latest.json"
    if latest_path.exists() and json.loads(latest_path.read_text(encoding="utf-8"))["episode"] == session["state"]["episode"]:
        backup_results(session["config"])
    return result

def refresh_test_summary(root):
    root = Path(root)
    summary = json.loads((root / "summary.json").read_text(encoding="utf-8"))
    tests = [json.loads(path.read_text(encoding="utf-8")) for path in sorted(root.glob("test_*.json"))]
    extra_validation = [json.loads(path.read_text(encoding="utf-8")) for path in sorted(root.glob("validation_*.json"))]
    full_tests = [json.loads(path.read_text(encoding="utf-8")) for path in sorted(root.glob("full_test_*.json"))]
    scores = [result["exact_match_accuracy"] for result in tests]
    summary["final_held_out_accuracy"] = scores[0] if scores and all(score == scores[0] for score in scores) else None
    f1_scores = [result.get("answer_f1") for result in tests]
    summary["final_held_out_f1"] = f1_scores[0] if f1_scores and all(score == f1_scores[0] for score in f1_scores) else None
    mr_scores = [result.get("mean_reward") for result in tests]
    summary["final_held_out_mean_reward"] = mr_scores[0] if mr_scores and all(score == mr_scores[0] for score in mr_scores) else None
    summary["test_status"] = "available" if summary["final_held_out_accuracy"] is not None else "repeated tests disagree" if scores else "not run"
    validation = read_jsonl(root / "validation.jsonl")
    evaluations = validation + tests + extra_validation + full_tests
    summary["evaluation_llm_calls"] = sum(row["execution_llm_calls"] for row in evaluations)
    summary["evaluation_pipeline_evaluations"] = sum(row["attempted_pipeline_evaluations"] for row in evaluations)
    summary["evaluation_seconds"] = sum(row["seconds"] for row in evaluations)
    summary["total_llm_calls_in_effective_history"] = summary["execution_llm_calls"] + summary["evaluation_llm_calls"]
    journal = read_jsonl(root / "attempts.jsonl") if (root / "attempts.jsonl").exists() else []
    attempts = [row for row in journal if row["kind"] in ("training_attempt", "evaluation_attempt")]
    summary["resource_journal_complete"] = all("resources" in row for row in attempts)
    write_json(root / "summary.json", summary, overwrite=True)
    return summary

def experiment_summary(session):
    state, cfg = session["state"], session["config"]; rows = state["train"]
    attempts = sum(r["attempted_pipeline_evaluations"] for r in rows)
    latest_path = session["run_dir"] / "latest.json"
    durable = json.loads(latest_path.read_text(encoding="utf-8"))["episode"] if latest_path.exists() else 0
    tests = []
    for path in sorted(session["run_dir"].glob("test_*.json")):
        tests.append(dict(path=str(path), **json.loads(path.read_text(encoding="utf-8"))))
    scores = [t["exact_match_accuracy"] for t in tests]
    final_accuracy = scores[0] if scores and all(score == scores[0] for score in scores) else None
    f1_scores = [t.get("answer_f1") for t in tests]
    final_f1 = f1_scores[0] if f1_scores and all(score == f1_scores[0] for score in f1_scores) else None
    test_status = "available" if scores else "not run"
    if scores and final_accuracy is None:
        test_status = "repeated tests disagree; inspect artifacts, no best-score selection"
    def total(key):
        values = [r[key] for r in rows]
        return sum(values) if all(v is not None for v in values) else None
    evaluation_calls = sum(v["execution_llm_calls"] for v in state["validation"]) + sum(t["execution_llm_calls"] for t in tests)
    return dict(training_candidates_reserved=reserved_training_candidates(session), candidate_budget=cfg["candidate_budget"],
                budget_exhausted=state.get("budget_exhausted", False), config=cfg, selection_id=session["data"]["selection_id"],
                initial_policy_digest=session["initial_policy_digest"],
                execution_model_digest=session["llm_metadata"].get("model", {}).get("digest"),
                runtime_identity=dict(packages=VERSIONS, ollama_version=session["llm_metadata"].get("version")), run_dir=str(session["run_dir"]),
                completed=(state["episode"] == cfg["num_episodes"] or state.get("budget_exhausted", False)) and durable == state["episode"] and any(row["episode"] == state["episode"] for row in state["validation"]),
                episode=state["episode"], durable_episode=durable, optimizer_updates=state["updates"],
                unique_training_questions=len(state["seen"]), best=state["best"], test_status=test_status,
                final_held_out_accuracy=final_accuracy, final_held_out_f1=final_f1,
                training_time=state["cumulative_training_seconds"],
                attempted_pipeline_evaluations=attempts, successful_pipeline_evaluations=total("successful_pipeline_evaluations"),
                failed_pipeline_evaluations=total("failed_pipeline_evaluations"), execution_llm_calls=total("execution_llm_calls"),
                physical_execution_llm_calls=total("physical_lm_calls"),
                cached_pipeline_executions=total("cached_pipeline_executions"),
                zero_variance_skipped_updates=sum(not r.get("optimizer_step", True) for r in rows),
                average_unique_pipeline_ratio=float(np.mean([r.get("unique_pipeline_ratio", 1.0) for r in rows])) if rows else None,
                prompt_tokens=total("prompt_tokens"), generated_tokens=total("generated_tokens"), physical_http_requests=None,
                training_mean_reward=float(np.mean([r["mean_group_reward"] for r in rows])) if rows else None,
                training_reward_variance=float(np.var([x for r in rows for x in r["rewards"]])) if rows else None,
                zero_advantage_group_percentage=100 * float(np.mean([r["zero_advantage_group"] for r in rows])) if rows else None,
                average_pipeline_execution_latency=sum(r["pipeline_execution_seconds"] for r in rows) / attempts if attempts else None)


# 19. Compare Adaptive-K and Static-K

After executing the definition cells, pass explicit completed run directories:

```python
runs = compare_runs([STATIC_RUN_DIR, ADAPTIVE_RUN_DIR],
                    comparison_axis="budget", candidate_cap=1000)
```

The result reports training attempts for each run; a common cap does not mean
both used the same number of candidates. Static journals record completed attempts,
so an in-flight interruption can be unmeasured. Validation and test calls are not
included in this cap. Comparisons check dataset, seed, model and runtime identities.
Both `bounded-rescue-v1` and `apex-adaptive-rollout-v2` artifact schemas are verified.


In [ ]:
def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]

def verify_adaptive_artifacts(root, summary):
    """Reject inconsistent histories before plotting; never rewrite source artifacts."""
    root = Path(root); cfg = summary["config"]
    train = read_jsonl(root / "training.jsonl"); validation = read_jsonl(root / "validation.jsonl")
    def require(condition, message):
        if not condition:
            raise ValueError(f"Inconsistent run artifacts at {root}: {message}")
    require(cfg.get("allocation_version") in ("bounded-rescue-v1", "apex-adaptive-rollout-v2"), "unsupported allocation")
    final_episode = summary["episode"]
    require(final_episode > 0 and final_episode == summary["durable_episode"], "incomplete durable history")
    if final_episode != cfg["num_episodes"]:
        require(summary.get("budget_exhausted") is True and cfg.get("candidate_budget") is not None, "unjustified early stop")
        journal = read_jsonl(root / "attempts.jsonl")
        reservations = sum(row.get("kind") == "training_reservation" for row in journal)
        require(reservations == summary.get("training_candidates_reserved") and 0 <= cfg["candidate_budget"] - reservations < cfg["adaptive_levels"][0], "invalid budget exhaustion")
    require([row["episode"] for row in train] == list(range(1, final_episode + 1)), "missing/duplicate training episodes")
    cumulative = 0; seen = set()
    for row in train:
        require((row["dataset"], row["seed"]) == (cfg["dataset"], cfg["seed"]) and row["k"] in cfg["adaptive_levels"], "episode identity")
        require(row["attempted_pipeline_evaluations"] == row["k"], "candidate budget")
        require(row["successful_pipeline_evaluations"] + row["failed_pipeline_evaluations"] == row["k"], "candidate outcomes")
        require(len(row["rewards"]) == len(row["advantages"]) == len(row["actions"]) == len(row["signatures"]) == row["k"], "candidate records")
        mean, advantages = group_advantages(row["rewards"])
        require(np.isclose(row["mean_group_reward"], mean) and np.allclose(row["advantages"], advantages), "centered reward statistics")
        require(row["zero_advantage_group"] == bool(np.all(advantages == 0)), "zero-advantage flag")
        cumulative += row["k"]; seen.add(row["question_id"])
        require(row["unique_training_questions"] == len(seen), "unique question count")
        if cfg.get("allocation_version") == "apex-adaptive-rollout-v2":
            require(row.get("optimizer_step") == (not row["zero_advantage_group"]), "optimizer step masking consistency")
        else:
            require(row["optimizer_update_count"] == row["episode"], "optimizer update count")
    require(summary["attempted_pipeline_evaluations"] == cumulative, "summary training budget")
    expected_updates = sum(r.get("optimizer_step", True) for r in train) if cfg.get("allocation_version") == "apex-adaptive-rollout-v2" else len(train)
    require(summary["optimizer_updates"] == expected_updates and summary["unique_training_questions"] == len(seen), "summary updates/questions")
    for key in ("successful_pipeline_evaluations", "failed_pipeline_evaluations", "execution_llm_calls"):
        require(summary[key] == sum(row[key] for row in train), "summary " + key)
    require(np.isclose(summary["training_time"], sum(row["episode_seconds"] for row in train)), "summary training time")
    expected = [0] + [episode for episode in range(1, cfg["num_episodes"] + 1)
                      if episode % cfg["validation_interval"] == 0 or episode == cfg["num_episodes"]]
    require([row["episode"] for row in validation] == expected, "validation progression")
    validation_ids = validation[0]["question_ids"]
    require(len(validation_ids) == len(set(validation_ids)) == cfg["validation_size"], "validation question IDs")
    for row in validation:
        require(row["question_ids"] == validation_ids and not seen.intersection(row["question_ids"]), "validation identity/leakage")
        require(row["questions"] == row["attempted_pipeline_evaluations"] == cfg["validation_size"], "validation pipeline budget")
        require(row["successful_pipeline_evaluations"] + row["failed_pipeline_evaluations"] == row["questions"], "validation outcomes")
    metric = "exact_match_accuracy"
    selected = max(validation, key=lambda row: (row[metric], -row["episode"]))
    best = summary.get("best")
    require(best is not None and best["episode"] == selected["episode"] and np.isclose(best["score"], selected[metric]), "validation-only checkpoint selection")
    tests = [json.loads(path.read_text(encoding="utf-8")) for path in sorted(root.glob("test_*.json"))]
    for result in tests:
        require(result["split"] == "test" and result["questions"] == cfg["test_size"], "held-out test scope")
        require(Path(result["selected_checkpoint"]).resolve() == Path(best["checkpoint"]).resolve(), "held-out checkpoint provenance")
        require(len(result["question_ids"]) == len(set(result["question_ids"])) == cfg["test_size"], "held-out question IDs")
        require(not (seen | set(validation_ids)).intersection(result["question_ids"]), "held-out leakage")
    scores = [row[metric] for row in tests]
    accuracy = scores[0] if scores and all(score == scores[0] for score in scores) else None
    require(summary["final_held_out_accuracy"] == accuracy, "held-out summary score")
    latest = json.loads((root / "latest.json").read_text(encoding="utf-8"))
    require(latest["episode"] == cfg["num_episodes"] and Path(latest["checkpoint"]).is_file(), "latest durable checkpoint")
    require(json.loads((root / "best.json").read_text(encoding="utf-8")) == best and Path(best["checkpoint"]).is_file(), "selected checkpoint pointer")
    metadata_path = root / "config.json"
    require(metadata_path.is_file(), "missing configuration snapshot")
    return dict(train=train, validation=validation)

def compare_runs(run_dirs, comparison_axis="episodes", candidate_cap=None):
    if not run_dirs: raise ValueError("Provide explicit completed run directories")
    runs = []; reference = None; seen = set()
    for root in map(Path, run_dirs):
        if root.resolve() in seen: raise ValueError("Duplicate run directory")
        seen.add(root.resolve())
        summary = json.loads((root / "summary.json").read_text(encoding="utf-8"))
        if not summary["completed"]: raise ValueError("Comparison requires complete runs")
        history = verify_adaptive_artifacts(root, summary)
        runs.append(dict(summary=summary, **history))
    return runs

def adaptive_results(session):
    rows = session["state"]["train"]
    return dict(
        episodes=len(rows),
        candidates=sum(r["attempted_pipeline_evaluations"] for r in rows),
        realized_k={k: sum(r["k"] == k for r in rows) for k in session["config"]["adaptive_levels"]},
        k_ceilings={k: sum(r.get("k_ceiling") == k for r in rows) for k in session["config"]["adaptive_levels"]},
        physical_calls=sum(r.get("physical_llm_calls", r["attempted_pipeline_evaluations"]) for r in rows),
        cached_executions=sum(r.get("cached_pipeline_executions", 0) for r in rows),
        optimizer_steps=sum(r.get("optimizer_step", True) for r in rows),
        zero_variance_skips=sum(r.get("zero_advantage_group", False) for r in rows),
        stop_reasons={reason: sum(r.get("stop_reason") == reason for r in rows)
                      for reason in ("contrast", "all_correct_exit", "ceiling_reached", "maximum_k",
                                    "failure_guard", "non_failure_homogeneous", "budget_exhausted")}
    )


# 20. Adaptive-K visualization


In [ ]:
def plot_adaptive_results(session):
    data = adaptive_results(session)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    
    # 1. Realized K vs Pre-allocated Ceiling
    labels = [str(k) for k in session["config"]["adaptive_levels"]]
    x = np.arange(len(labels))
    width = 0.35
    axes[0].bar(x - width/2, [data["realized_k"].get(k, 0) for k in session["config"]["adaptive_levels"]], width, label="Realized K", color="tab:blue")
    axes[0].bar(x + width/2, [data["k_ceilings"].get(k, 0) for k in session["config"]["adaptive_levels"]], width, label="Ceiling (Pillar 1)", color="tab:cyan", alpha=0.7)
    axes[0].set(xlabel="Group Size K", ylabel="Episodes", title="Group Size vs Ceiling Allocation")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels)
    axes[0].legend()
    
    # 2. Physical LLM Calls vs Cached Deduplicated Calls
    phys = data["physical_calls"]
    cached = data["cached_executions"]
    axes[1].bar(["Physical Ollama", "Cached (Deduplicated)"], [phys, cached], color=["tab:orange", "tab:green"])
    axes[1].set(ylabel="LM Evaluations", title=f"Deduplication Savings ({cached}/{phys+cached} saved)")
    
    # 3. Informative Updates vs Masked Zero-Variance Steps
    updates = data["optimizer_steps"]
    skips = data["zero_variance_skips"]
    axes[2].bar(["Optimizer Step", "Zero-Variance Masked"], [updates, skips], color=["tab:purple", "tab:gray"])
    axes[2].set(ylabel="Episodes", title=f"Pillar 4 Masking ({skips} drift events prevented)")
    
    fig.tight_layout()
    return fig


# 21. Reproducibility and Experiment Summary
Keep `RUN_OPERATION=False` when running definition cells. Rebuild CONFIG after edits and explicitly enable the operation below when ready. Training does not automatically test. `RUN_ALL_K=True` requests sequential independent training; each policy is released before the next.

Resume/validate/test require SELECTED_CHECKPOINT. For resume use latest.json; for final testing use best.json. Remote inference is not guaranteed deterministic. First-token collisions, last-number extraction, failure rewards and K-dependent gradient scaling remain documented limitations.

Notebook-local synthetic checks below use tiny test policies and mocks, never a substituted research model. They verify mechanics, never generate benchmark results. See [usage guide](STATIC_K_EXPERIMENT_GUIDE.md).

In [ ]:
def run_operation(cfg,mode,checkpoint=None,run_dirs=None,all_k=False):
    if mode=="compare":
        return compare_runs(run_dirs or [])
    if mode not in ("initialize","train","resume","validate","test"):
        raise ValueError("MODE must be initialize/train/resume/validate/test/compare")
    if all_k:
        raise ValueError("RUN_ALL_K is unavailable for adaptive runs")
    if all_k:
        if mode!="train":
            raise ValueError("RUN_ALL_K requires MODE=train")
        return run_all_k_sequential(cfg)
    if mode in ("resume","validate","test"):
        if checkpoint is None:
            raise ValueError("Set SELECTED_CHECKPOINT to an explicit .pt path")
        session=load_selected_experiment(checkpoint,cfg,for_resume=mode=="resume")
    else:
        session=initialize_experiment(cfg)
    if mode in ("train","resume"):
        train_grpo(session)
    elif mode=="validate":
        validate_selected(session)
    elif mode=="test":
        test_model(session)
    return session

COMPARISON_RUN_DIRS=[]
if RUN_OPERATION:
    CONFIG=build_config()
    if "SESSION" in globals() and isinstance(SESSION,dict) and "policy" in SESSION:
        release_experiment(SESSION)
    SESSION=run_operation(CONFIG,MODE,checkpoint=SELECTED_CHECKPOINT,
                          run_dirs=COMPARISON_RUN_DIRS,all_k=RUN_ALL_K)
else:
    print("Setup complete. No training/test launched. Run adaptive_self_checks() for synthetic controller checks.")

In [ ]:
def adaptive_self_checks():
    print("Running APEX-GRPO 4-Pillar and HotpotQA Loss Normalization self-checks...")
    cfg = dict(adaptive_levels=[5, 8, 10, 15], rescue_mode="failure_only", reward_contrast=0.0, lambda_penalty=0.25)

    # 1. HotpotQA Answer Normalization & Metrics Check
    raw = "The Scott Family Park!"
    norm = normalize_answer(raw)
    assert norm == "scott family park", f"Expected 'scott family park', got '{norm}'"
    f1, prec, rec = answer_f1("Scott Family Park", "the scott family park.")
    assert np.isclose(f1, 1.0) and np.isclose(prec, 1.0) and np.isclose(rec, 1.0)

    # 2. Test Pillar 1: Analytical Pre-Allocation & Utility Optimization
    uniform_p = np.ones(30) / 30.0
    k_ceil_u, cov_u, ut_u, ent_u = solve_preallocation_ceiling(uniform_p, cfg)
    assert k_ceil_u == 15, f"Uniform policy should permit K=15, got {k_ceil_u}"

    collapsed_p = np.zeros(30)
    collapsed_p[0] = 0.99
    collapsed_p[1:] = 0.01 / 29.0
    k_ceil_c, cov_c, ut_c, ent_c = solve_preallocation_ceiling(collapsed_p, cfg)
    assert k_ceil_c == 5, f"Collapsed policy must be capped at K=5, got {k_ceil_c}"

    k_ceil_reg, _, _, _ = solve_preallocation_ceiling(uniform_p, cfg, historical_pass_rate=(0.0, 3))
    assert k_ceil_reg == 5, f"Consistently unsolvable prompt must be regularized to K=5, got {k_ceil_reg}"

    # 3. Test Pillar 3: Staged Waterfall Stopping Decisions
    assert adaptive_decision([0]*4 + [1], False, cfg, k_ceiling=15) == "contrast"
    assert adaptive_decision([0]*7 + [1], False, cfg, k_ceiling=15) == "contrast"
    assert adaptive_decision([1]*5, False, cfg, k_ceiling=15) == "all_correct_exit"
    assert adaptive_decision([0]*5, False, cfg, k_ceiling=5) == "ceiling_reached"
    assert adaptive_decision([0]*5, False, cfg, k_ceiling=8) is None
    assert adaptive_decision([0]*8, False, cfg, k_ceiling=8) == "ceiling_reached"
    assert adaptive_decision([0]*8, False, cfg, k_ceiling=10) is None
    assert adaptive_decision([0]*15, False, cfg, k_ceiling=15) == "ceiling_reached"
    assert adaptive_decision([0]*5, True, cfg, k_ceiling=15) == "failure_guard"

    # 4. Test Advantages & 1/K Loss Normalization
    mean, adv = group_advantages([0, 0, 1])
    assert np.allclose(adv, [-1/3, -1/3, 2/3])
    dummy_log_probs = [[torch.tensor(-1.0)], [torch.tensor(-2.0)], [torch.tensor(-3.0)]]
    dummy_adv = [-1/3, -1/3, 2/3]
    loss_val = figure2_loss(dummy_log_probs, dummy_adv)
    assert np.isclose(float(loss_val), 1.0 / 3.0), f"Loss should be normalized by 1/K: got {float(loss_val)}"

    # 5. Test Zero-Variance Group (Pillar 4 Masking Check)
    mean_zero, adv_zero = group_advantages([0, 0, 0, 0, 0])
    assert np.all(adv_zero == 0.0)
    mean_pass, adv_pass = group_advantages([1, 1, 1, 1, 1])
    assert np.all(adv_pass == 0.0)

    print("All APEX-GRPO 4-Pillar and HotpotQA Loss Normalization self-checks passed successfully!")


## Explicit operations

Set one switch to `True` and run its cell. Reset it to `False` afterward. A baseline run has 200 episodes, validation at 0/50/100/150/200, and checkpoints every 10 episodes. A finite candidate budget can stop earlier; no episode-200 checkpoint then exists. Test evaluations use the same held-out subset and should be reported separately; do not select a checkpoint using test scores.

In [ ]:
# Fresh run. This can take a long time; it does not run until enabled.
START_NEW_TRAINING = False
if START_NEW_TRAINING:
    if "SESSION" in globals() and isinstance(SESSION, dict) and "policy" in SESSION:
        release_experiment(SESSION)
    CONFIG = build_config()
    SESSION = run_operation(CONFIG, "initialize")
    RUN_DIR = Path(SESSION["run_dir"])
    print("Run directory:", RUN_DIR)
    train_grpo(SESSION)
else:
    print("Fresh training disabled. Set START_NEW_TRAINING = True to start.")

In [ ]:
# Resume the latest durable checkpoint of an interrupted run.
RESUME_TRAINING = False
RUN_DIR = globals().get("RUN_DIR", None)  # Or set Path("/content/adaptive_grpo_results/.../run-id")
if RESUME_TRAINING:
    if RUN_DIR is None:
        raise ValueError("Set RUN_DIR to the restored run directory")
    RUN_DIR = Path(RUN_DIR)
    CONFIG = json.loads((RUN_DIR / "config.json").read_text(encoding="utf-8"))["config"]
    checkpoint = json.loads((RUN_DIR / "latest.json").read_text(encoding="utf-8"))["checkpoint"]
    if "SESSION" in globals() and isinstance(SESSION, dict) and "policy" in SESSION:
        release_experiment(SESSION)
    SESSION = load_selected_experiment(checkpoint, CONFIG, for_resume=True)
    print("Resuming run directory:", SESSION["run_dir"])
    train_grpo(SESSION)
else:
    print("Resume disabled.")

In [ ]:
# Read validation results without running inference.
if RUN_DIR is not None:
    from IPython.display import display
    RUN_DIR = Path(RUN_DIR)
    validation_results = read_jsonl(RUN_DIR / "validation.jsonl")
    display([{key: row.get(key) for key in ("episode", "questions", "exact_match_accuracy", "answer_f1", "mean_reward", "failed_pipeline_evaluations")} for row in validation_results])
    print("Best checkpoint:")
    display(json.loads((RUN_DIR / "best.json").read_text(encoding="utf-8")))
else:
    print("Set RUN_DIR to inspect a run.")


In [ ]:
def load_for_checkpoint_evaluation(run_dir, checkpoint):
    global SESSION
    run_dir = Path(run_dir)
    cfg = json.loads((run_dir / "config.json").read_text(encoding="utf-8"))["config"]
    if "SESSION" in globals() and isinstance(SESSION, dict) and "policy" in SESSION:
        release_experiment(SESSION)
    SESSION = load_selected_experiment(checkpoint, cfg, for_resume=False)
    return SESSION

TEST_BEST = False
if TEST_BEST:
    if RUN_DIR is None:
        raise ValueError("Set RUN_DIR first")
    RUN_DIR = Path(RUN_DIR)
    best = json.loads((RUN_DIR / "best.json").read_text(encoding="utf-8"))
    best_session = load_for_checkpoint_evaluation(RUN_DIR, best["checkpoint"])
    BEST_TEST = test_model(best_session)
    backup_results(best_session["config"])
    print(f"Best checkpoint held-out EM: {BEST_TEST['exact_match_accuracy']*100:.1f}% | "
          f"F1: {BEST_TEST['answer_f1']:.3f} | Mean Reward (Semantic): {BEST_TEST['mean_reward']:.3f} ({BEST_TEST['mean_reward']*100:.1f}%)")
else:
    print("Best-checkpoint test disabled.")


In [ ]:
# Optional separate test of the fixed episode-200 checkpoint. This does not
# replace the validation-selected best checkpoint or the best-test summary.
TEST_EPISODE_200 = False
if TEST_EPISODE_200:
    if RUN_DIR is None:
        raise ValueError("Set RUN_DIR first")
    RUN_DIR = Path(RUN_DIR)
    latest = json.loads((RUN_DIR / "latest.json").read_text(encoding="utf-8"))
    summary = json.loads((RUN_DIR / "summary.json").read_text(encoding="utf-8"))
    if not summary["completed"] or latest["episode"] != 200:
        raise ValueError("This operation requires a completed run whose latest checkpoint is episode 200")
    final_session = load_for_checkpoint_evaluation(RUN_DIR, latest["checkpoint"])
    EPISODE_200_TEST = evaluate_module(final_session, "test")
    EPISODE_200_TEST["checkpoint_selection_rule"] = "fixed episode 200; not selected by held-out results"
    final_test_path = RUN_DIR / f"episode_200_test_{uuid.uuid4().hex[:8]}.json"
    write_json(final_test_path, EPISODE_200_TEST)
    backup_results(final_session["config"])
    print(f"Episode-200 held-out EM: {EPISODE_200_TEST['exact_match_accuracy']*100:.1f}% | "
          f"F1: {EPISODE_200_TEST['answer_f1']:.3f} | Mean Reward (Semantic): {EPISODE_200_TEST['mean_reward']:.3f} ({EPISODE_200_TEST['mean_reward']*100:.1f}%) | artifact: {final_test_path}")
else:
    print("Episode-200 test disabled.")


### Plots for the best and episode-200 checkpoints

These figures read saved logs only. Each chart shows the complete recorded training history and marks its selected checkpoint; a held-out score is shown only if the corresponding explicit test has been run. The episode-200 figure requires an actual latest episode-200 checkpoint. Files are written under a new timestamped run-local `plots/` directory and backed up to Drive.

In [ ]:
def plot_checkpoint_figures(run_dir):
    run_dir = Path(run_dir)
    train = read_jsonl(run_dir / "training.jsonl")
    validation = read_jsonl(run_dir / "validation.jsonl")
    if not train or not validation:
        raise ValueError("Training and validation records are required")
    summary = json.loads((run_dir / "summary.json").read_text(encoding="utf-8"))
    best = json.loads((run_dir / "best.json").read_text(encoding="utf-8"))
    latest = json.loads((run_dir / "latest.json").read_text(encoding="utf-8"))
    checkpoints = [("best_checkpoint", best)]
    if latest["episode"] == 200 and Path(latest["checkpoint"]).is_file():
        checkpoints.append(("episode_200", latest))
    else:
        print("Episode-200 figure skipped: no latest episode-200 checkpoint.")
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
    output_dir = run_dir / "plots" / stamp
    output_dir.mkdir(parents=True, exist_ok=False)
    episodes = [row["episode"] for row in train]
    for label, pointer in checkpoints:
        checkpoint = Path(pointer["checkpoint"])
        if not checkpoint.is_file():
            raise FileNotFoundError(checkpoint)
        pattern = "test_*.json" if label == "best_checkpoint" else "episode_200_test_*.json"
        tests = [json.loads(p.read_text(encoding="utf-8")) for p in sorted(run_dir.glob(pattern))]
        tests = [t for t in tests if t.get("split") == "test" and t.get("episode") == pointer["episode"]
                 and t.get("selected_checkpoint") and Path(t["selected_checkpoint"]).resolve() == checkpoint.resolve()
                 and t.get("questions") == summary["config"]["test_size"]]
        if len({t["exact_match_accuracy"] for t in tests}) > 1:
            raise ValueError(f"Repeated {label} tests disagree; inspect saved test artifacts")
        fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
        ax = axes.flat
        ax[0].plot([v["episode"] for v in validation], [100*v["exact_match_accuracy"] for v in validation], marker="o", label="Exact Match (%)")
        if any("answer_f1" in v for v in validation):
            ax[0].plot([v["episode"] for v in validation], [100*v["answer_f1"] for v in validation], marker="^", linestyle=":", label="Answer F1 (%)")
        if any("mean_reward" in v for v in validation):
            ax[0].plot([v["episode"] for v in validation], [100*v["mean_reward"] for v in validation], marker="s", linestyle="--", label="Mean Reward (Semantic %)")
        ax[0].set(title="Validation Performance", xlabel="Episode", ylabel="Score (%)", ylim=(0, 100))
        ax[0].legend(loc="best", fontsize="small")
        ax[1].plot(episodes, [r["mean_group_reward"] for r in train])
        ax[1].set(title="Mean candidate reward", xlabel="Episode", ylabel="Reward")
        ax[2].plot(episodes, [r["gradient_norm"] for r in train])
        ax[2].set(title="Gradient norm", xlabel="Episode", ylabel="L2 norm")
        ax[3].step(episodes, [r["k"] for r in train], where="mid")
        ax[3].set(title="Realized adaptive group size", xlabel="Episode", ylabel="K")
        ax[4].plot(episodes, np.cumsum([r["attempted_pipeline_evaluations"] for r in train]))
        ax[4].set(title="Cumulative training pipeline evaluations", xlabel="Episode", ylabel="Pipeline evaluations")
        if tests:
            test_em = 100 * tests[0]["exact_match_accuracy"]
            test_f1 = 100 * tests[0].get("answer_f1", test_em)
            test_mr = 100 * tests[0].get("mean_reward", test_em)
            x_pos = np.arange(3)
            bars = ax[5].bar(x_pos, [test_em, test_f1, test_mr], color=["tab:blue", "tab:purple", "tab:orange"], width=0.5)
            ax[5].set_xticks(x_pos)
            ax[5].set_xticklabels(["Exact Match", "Answer F1", "Mean Reward\n(Semantic)"])
            ax[5].set(title=f"{label}: test (n={tests[0]['questions']})", ylabel="Score (%)", ylim=(0, 100))
            for b, val in zip(bars, [test_em, test_f1, test_mr]):
                ax[5].text(b.get_x() + b.get_width()/2, val + 1.5, f"{val:.1f}%", ha="center", fontsize="small", fontweight="bold")
        else:
            ax[5].axis("off")
            ax[5].text(.5, .5, "Held-out test not run", ha="center", va="center", transform=ax[5].transAxes)
        for a in ax[:5]:
            a.axvline(pointer["episode"], color="tab:red", linestyle="--", label=f"Checkpoint: episode {pointer['episode']}")
            a.legend(loc="best", fontsize="small")
        fig.suptitle(f"HotpotQA Adaptive-K | {label} | seed {summary['config']['seed']} | full recorded training history")
        for extension in ("png", "pdf"):
            fig.savefig(output_dir / f"{label}.{extension}", dpi=200, bbox_inches="tight")
        plt.show()
        plt.close(fig)
    backup_results(summary["config"])
    print("Figures:", output_dir)
    return output_dir

GENERATE_PLOTS = False
if GENERATE_PLOTS:
    if RUN_DIR is None:
        raise ValueError("Set RUN_DIR first")
    PLOT_DIR = plot_checkpoint_figures(RUN_DIR)
else:
    print("Plot generation disabled.")
